# 03 · NeuroLoc-Δ nel cervello: probabilità tarate, percentuali, varianti e malattie neurologiche

Lo 02 ha scelto e valutato il modello sul test HPA indipendente. Questo notebook trasforma le sue uscite nello
strumento per chi studia le malattie neurologiche:
1. **prior dalla sequenza** per ogni proteina umana del cervello (out-of-fold per il training, media dei fold per il test);
2. **contesto dei partner** (CORUM, OpenCell) per tutti i 14 compartimenti, sinapsi e mielina comprese, e **calibrazione**:
   una "probabilità del 70%" deve essere vera 7 volte su 10;
3. **fusione con le misure** già esistenti (LOPIT, BioID, OpenCell, HPA, UniProt, SynGO, MitoCarta), con le fonti dello
   stesso tipo contate una volta e la calibrazione verificata dopo la fusione;
4. **ripartizioni %** (citosol solubile e nucleo misurati o predetti, il resto tra gli organelli);
5. la **tabella del cervello**;
6. le **varianti**: un punteggio di priorità per patogenicità e, dichiarato, un **risultato negativo**: i segnali di
   spostamento della localizzazione non predicono la mislocalizzazione misurata;
7. le **malattie**: in quali compartimenti cadono le proteine delle malattie neurologiche, e alcuni casi di studio;
8. le **curve ROC finali** dei compartimenti neuronali e delle varianti;
9. le **proteine nuove** (sezione 6b): una sequenza che non è nei dati passa per lo stesso modello e gli stessi passaggi;
10. la **pagina HTML locale** `data/neuroloc_delta.html` (sezione 11): ricerca su tutte le proteine, scheda completa di
    ognuna con le card_protein delle proteine più simili, e un pannello per incollare una sequenza nuova direttamente nel
    browser.
11. il **contesto di Wefers et al. 2026** (sezione 12): la rete di sola sequenza sulle proteine del loro test HOU, fuori
    fold, accanto ai valori pubblicati. Non è un confronto appaiato: quasi tutte le proteine HOU sono nei nostri dati.

Gira su CPU in pochi minuti (con proteine nuove servono anche ESM-2 e ProtT5, meglio su GPU). Input: `data/esiti_02.npz`, `data/esiti_02.json`, `data/proteine.parquet`,
`data/grafo.parquet`, `data/varianti_codificate.parquet`, `data/descrittori_sequenza.parquet`,
`data/varianti_sperimentali.parquet`, i file per-residuo di ESM-2 (`data/res_*`) e le strutture AlphaFold (`data/alphafold`).

## Nono giro: domande da neurologo e da biologo cellulare
49. *Quali difetti di localizzazione contano nelle malattie del sistema nervoso?* Sei meccanismi ricorrono, e ognuno ha
    un segnale che il modello vede:
    | meccanismo | esempi | cosa guarda il modello |
    |---|---|---|
    | trasporto nucleo-citoplasma | TDP-43 citoplasmatica nella SLA e nella FTD; mutazioni del PY-NLS di FUS (R521C, P525L) | testa NLS, P(nucleo), domini RGG |
    | ritenzione nel RE di proteine mal ripiegate | PLP1 (Pelizaeus-Merzbacher), canali ionici delle epilessie (SCN1A, KCNQ2) | via secretoria × destabilizzazione (LLR di ESM) × pLDDT del sito |
    | targeting in assone e segmento iniziale | legame all'ankirina-G dei canali Nav e KCNQ; tau che finisce nel compartimento somatodendritico nell'Alzheimer | motivo ankG, Δ assone/dendrite |
    | ancoraggio sinaptico | ligandi PDZ C-terminali (GluN2B, neuroligine) persi | perdita del ligando PDZ, Δ postsinapsi |
    | import negli organelli | presequenza di PINK1 (Parkinson); PTS1 dei perossisomi (spettro di Zellweger, leucodistrofie) | momento idrofobico N-terminale, PTS1/PTS2 |
    | ancore lipidiche | palmitoilazione di PSD-95; ZDHHC9 nella disabilità intellettiva | cisteine N-terminali, testa della palmitoilazione |
50. *Perché i compartimenti neuronali solo per le proteine dei neuroni?* "Presente nell'assone" non ha senso per una
    proteina che il neurone non produce; la mielina è la membrana degli oligodendrociti. Le colonne sono vuote fuori
    dalle cellule giuste, invece di mostrare una probabilità priva di significato.
51. *I partner aiutano anche per la sinapsi?* Più che per gli organelli. La densità postsinaptica è un'impalcatura
    (PSD-95, SHANK, Homer, GKAP): una proteina legata a PSD-95 sta quasi sempre alla postsinapsi, anche se la sequenza
    non lo dice. Il contesto dei partner ora copre tutti e 14 i compartimenti, sempre con le sole etichette fuori fold.
52. *Come si valuta un punteggio di mislocalizzazione senza un laboratorio?* In due modi, con limiti diversi. ClinVar
    (patogeniche contro benigne) misura la patogenicità, non la mislocalizzazione: è un banco di prova indiretto. Le
    varianti con localizzazione misurata (Lacoste et al. 2024, scaricate dallo 01) sono il banco diretto, e lì i segnali
    di spostamento non funzionano (sezione 7): è un risultato negativo, riportato con gli intervalli di confidenza.
53. *Il punteggio supervisionato su ClinVar può barare?* Sì, in due modi già visti in letteratura: la quota di varianti
    patogeniche del gene (nella versione precedente era una feature: conteneva le etichette del gene stesso) e varianti
    dello stesso gene in training e in test. Qui la feature è tolta e la CV è raggruppata per proteina.
54. *Le proteine delle malattie stanno dove la biologia si aspetta?* È il controllo di buon senso dello strumento: le
    epilessie sono soprattutto canalopatie (membrana, presinapsi), le leucodistrofie colpiscono mielina, perossisomi e
    lisosomi, le disabilità intellettive coinvolgono regolatori della cromatina (nucleo) e la postsinapsi. Se le mappe
    non ritrovano queste associazioni, qualcosa non va. Il controllo usa le probabilità della sola sequenza, fuori fold:
    con quelle fuse con le misure il risultato ripeterebbe le annotazioni.

In [ ]:
import json
import re
import warnings
from itertools import compress
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm, spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold, cross_val_predict

DATA = Path('data')
plt.style.use('default')                      # figure su sfondo bianco, pronte per il paper
SEED = 42
ESCLUDI_SANGUE = True
warnings.filterwarnings('ignore', message='Mean of empty slice')
rng = np.random.default_rng(SEED)
cfg = json.loads((DATA / 'esiti_02.json').read_text(encoding='utf-8'))
z = np.load(DATA / 'esiti_02.npz')
proteins = pd.read_parquet(DATA / 'proteine.parquet')
grafo = pd.read_parquet(DATA / 'grafo.parquet')
cv = pd.read_parquet(DATA / 'varianti_codificate.parquet')
hand = pd.read_parquet(DATA / 'descrittori_sequenza.parquet')
lab_var = pd.read_parquet(DATA / 'varianti_sperimentali.parquet')
CLASSES, NEURO, ALL, SOURCES, AUX = (cfg[k] for k in ['CLASSES', 'NEURO', 'ALL', 'SOURCES', 'AUX'])
N_PROT, S, C, nC, A, WIN = len(proteins), len(SOURCES), len(ALL), len(CLASSES), len(AUX), cfg['WIN']
OBS = z['OBS'].astype(np.float32)
FRAC, part = z['FRAC'], z['part']
gold_rows, human_gold, test_rows, var_rows = z['gold_rows'], z['human_gold'], z['test_rows'], z['var_rows']
src = proteins['source'].to_numpy()
is_brain, is_neur = proteins['is_brain'].to_numpy().astype(bool), proteins['is_neuronal'].to_numpy().astype(bool)
is_oligo, is_blood = proteins['is_oligo'].to_numpy().astype(bool), proteins['is_blood'].to_numpy().astype(bool)
is_human = proteins['is_human'].to_numpy().astype(bool) | proteins['gene'].notna().to_numpy()
GOLD = OBS[:, 0, :nC]
gate = np.stack([is_brain] * nC + [is_neur] * (len(NEURO) - 1) + [is_oligo], 1)       # dove ogni compartimento ha senso
CTX = list(proteins.columns[proteins.columns.str.startswith('ctx_')]) + ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm']
proteins[CTX] = proteins[CTX].replace([np.inf, -np.inf], np.nan)                    # log10(0 copie) = −inf nel 01
print(f"Modello scelto nello 02: {cfg['BEST']} | proteine {N_PROT:,} | fonti {SOURCES}")

## 1 · Il prior dalla sequenza
Per le proteine di training ed estese si usa la predizione **out-of-fold** della rete sul pool esteso (stadio `cervello`
dello 02): ogni proteina è predetta da un modello che non l'ha vista. Le proteine del test HPA ricevono la media dei 5
modelli. Nello strumento finale le etichette HPA del test diventano misure della fonte `hpa_alta` (lo sono: vengono
dallo stesso atlante); nelle valutazioni non entrano mai.

In [ ]:
LOGIT = z['oof'].copy()                                        # [proteina, uscita]: 14 classi, ausiliari, 2 frazioni
LOGIT[test_rows] = z['hpa']
P_best = z['P_oof_best'].astype(np.float64).copy()
P_best[test_rows] = z['P_hpa_best']
lg_best = np.log(np.clip(P_best, 1e-6, 1 - 1e-6) / (1 - np.clip(P_best, 1e-6, 1 - 1e-6)))
LOGIT[:, :nC] = np.where(np.isfinite(lg_best), lg_best, LOGIT[:, :nC])
Pb = 1 / (1 + np.exp(-LOGIT))
has_pred = np.isfinite(Pb[:, 0])
# predizioni fuori fold: le uniche valutabili. Restano fuori le proteine aggiunte che hanno un'omologa curata in un altro
# fold (elenco dello 02): il modello che le predice ha visto la loro omologa
oof_ok = np.isfinite(z['oof'][:, 0]) & z['valutabile']
MIN_POS = 30                                                   # sotto 30 positivi di riferimento una classe non si valuta
OBS_APP = OBS.copy()
seen_hpa = np.delete(np.arange(nC), CLASSES.index('Extracellular'))                   # HPA non vede l'extracellulare
OBS_APP[np.ix_(test_rows, [SOURCES.index('hpa_alta')], seen_hpa)] = z['Y_test'][:, seen_hpa][:, None, :]
print(f'Proteine con predizione: {has_pred.sum():,} (fuori fold {oof_ok.sum():,}, test {len(test_rows):,})')

## 2 · Valutazione nel cervello (out-of-fold) e affidabilità delle fonti
Riferimento: per gli organelli le fonti forti (DeepLoc, UniProt sperimentale, HPA Enhanced/Supported) sulle proteine
cerebrali; per assone, dendrite, sinapsi e mielina qualsiasi fonte, sulle proteine delle cellule giuste. AUC e AUPRC
con la formula dei ranghi: NaN invece di errore se una classe non ha positivi.

**Come leggere le classi neuronali (limite dichiarato).** Assone, dendrite, sinapsi e mielina non hanno etichette
curate né un test esterno: il riferimento sono le stesse fonti usate per l'addestramento (fuori fold), e i negativi
sono proteine "non annotate", non proteine "assenti". È un problema di positivi e non etichettati: l'AUC misura se il
modello ordina le annotate sopra le altre; l'AUPRC è una stima per difetto (una proteina vera ma non annotata conta
come errore) e le probabilità non sono frequenze. Una classe con meno di 30 positivi **non si valuta**: la mielina ne
ha una manciata e resta fuori da tabelle, curve e controlli.

Affidabilità delle fonti: accanto a sensibilità e specificità stimate dal canale di rumore c'è l'accordo **osservato**
con le etichette DeepLoc, sulle proteine che hanno entrambe (solo le 9 classi di DeepLoc). È una misura senza modello:
se il canale dà a una fonte una sensibilità molto più alta di quella osservata, la stima del canale è da non prendere
alla lettera. Le classi neuronali non hanno etichette curate e restano fuori da questo controllo.

In [ ]:
ref_src = np.isin(SOURCES, ['deeploc', 'uniprot_forte', 'hpa_alta'])
REF = np.where(np.isnan(OBS[:, ref_src]), -1, OBS[:, ref_src]).max(1)                # −1 = nessuna fonte di riferimento
REF[:, nC:] = np.where(np.isnan(OBS[:, :, nC:]), -1, OBS[:, :, nC:]).max(1)
masks = [oof_ok & (REF[:, c] >= 0) & gate[:, c] for c in range(C)]
with np.errstate(all='ignore'):
    report = pd.DataFrame({'proteine': [m.sum() for m in masks],
                           'positivi': [(REF[m, c] == 1).sum() for c, m in enumerate(masks)],
                           'AUC': [(pd.Series(Pb[m, c]).rank().to_numpy()[y].sum() - y.sum() * (y.sum() + 1) / 2) / (y.sum() * (~y).sum())
                                   for c, m in enumerate(masks) for y in [REF[m, c] == 1]],
                           'AUPRC': [(np.cumsum(y_s) / np.arange(1, len(y_s) + 1) * y_s).sum() / y_s.sum()
                                     for c, m in enumerate(masks) for y_s in [(REF[m, c] == 1)[np.argsort(-Pb[m, c])]]]}, index=ALL)
report['prevalenza (= AUPRC casuale)'] = report['positivi'] / report['proteine']
# una classe con pochi positivi non ha una valutazione: i numeri si tolgono invece di mostrarli come se dicessero qualcosa
VALUTABILE_C = (report['positivi'] >= MIN_POS).to_numpy()
report[['AUC', 'AUPRC']] = report[['AUC', 'AUPRC']].where(VALUTABILE_C[:, None])
report['valutazione'] = np.select([~VALUTABILE_C, np.arange(C) >= nC], ['non valutabile (< 30 positivi)', 'positivi contro non annotati'], 'etichette curate o forti')
print(report.round(3).to_string())
print(f"Proteine fuori dalle valutazioni perché hanno un'omologa curata in un altro fold: {int((~z['valutabile']).sum()):,}")

chan = z['chan']
seen = ~np.isnan(OBS).all(0)
reliab = pd.DataFrame({'sensibilità media': [chan[0, s][seen[s]].mean() for s in range(S)],
                       'specificità media': [chan[1, s][seen[s]].mean() for s in range(S)],
                       'classi misurate': seen.sum(1)}, index=SOURCES)
reliab.loc['deeploc', ['sensibilità media', 'specificità media']] = 1.0
# accordo diretto con DeepLoc: delle localizzazioni curate quante la fonte riporta (sensibilità), delle assenze curate
# quante conferma (specificità); media sulle classi, dove fonte e DeepLoc si sovrappongono
G9, O9 = OBS[:, :1, :nC], OBS[:, :, :nC]
both = ~np.isnan(G9) & ~np.isnan(O9)
with np.errstate(all='ignore'):
    reliab['sensibilità osservata (vs DeepLoc)'] = np.nanmean(((O9 == 1) & (G9 == 1)).sum(0) / ((G9 == 1) & both).sum(0), 1)
    reliab['specificità osservata (vs DeepLoc)'] = np.nanmean(((O9 == 0) & (G9 == 0)).sum(0) / ((G9 == 0) & both).sum(0), 1)
reliab['proteine in comune con DeepLoc'] = both.any(2).sum(0)
print('\nAffidabilità delle fonti: stimata dal canale di rumore e osservata contro DeepLoc:')
print(reliab.round(3).to_string())

## 3 · Partner molecolari per tutti i compartimenti e calibrazione
Per ogni proteina e compartimento: la quota dei partner (CORUM, OpenCell) localizzati lì e quanti partner hanno
un'etichetta. Per le proteine della partizione k contano solo etichette fuori da k; per il test tutte quelle di
training. Le fonti deboli (UniProt per similarità, HPA Approved, SynGO solo funzionale) non entrano.
Poi, per ogni classe, una logistica su `[logit della rete, quota dei partner, log(1 + partner etichettati)]`, stimata
in cross-fitting sulle proteine umane con riferimento: è insieme la **scala di Platt** (calibrazione) e il secondo
stadio del contesto (domanda 51). ECE a 10 intervalli prima e dopo.

In [ ]:
lab_src = ~np.isin(SOURCES, ['uniprot_debole', 'hpa_approved', 'syngo_funzionale'])
LAB = np.where(np.isnan(OBS[:, lab_src]), -1, OBS[:, lab_src]).max(1)
LAB = np.where(LAB < 0, np.nan, LAB)
NBL, NBN = np.full((N_PROT, C), np.nan, np.float32), np.zeros((N_PROT, C), np.float32)
for k in [0, 1, 2, 3, 4, -1]:
    e = grafo[part[grafo['pid_a'].to_numpy()] == k]
    g_ = pd.DataFrame(np.where((part[e['pid_b'].to_numpy()] != k)[:, None], LAB[e['pid_b'].to_numpy()], np.nan),
                      index=e['pid_a'].to_numpy()).groupby(level=0)
    NBL[g_.mean().index] = g_.mean().to_numpy()
    NBN[g_.count().index] = g_.count().to_numpy()

logit_b = np.log(np.clip(Pb[:, :C], 1e-6, 1 - 1e-6) / (1 - np.clip(Pb[:, :C], 1e-6, 1 - 1e-6)))
Pcal = Pb[:, :C].copy()
lab_cal = np.where(REF[:, :C] >= 0, REF[:, :C], np.nan)
ref_h = [m & is_human for m in masks]
n_ref = np.array([[(lab_cal[ref_h[c], c] == v).sum() for v in [0, 1]] for c in range(C)])
CAL_OK = np.flatnonzero(n_ref.min(1) >= 10)                    # le classi con meno di 10 positivi o negativi restano come sono
CAL_MODEL = {}                                                 # il modello stimato su tutto (k = −1) serve per le proteine nuove
for c in CAL_OK:
    Zc = np.c_[logit_b[:, c], np.nan_to_num(NBL[:, c]), np.log1p(NBN[:, c])]
    for k in [0, 1, 2, 3, 4, -1]:
        fit_c, app_c = ref_h[c] & (part != k), has_pred & (part == k)
        CAL_MODEL[c] = LogisticRegression(C=1.0, max_iter=1000).fit(Zc[fit_c], lab_cal[fit_c, c])
        Pcal[app_c, c] = CAL_MODEL[c].predict_proba(Zc[app_c])[:, 1]
with np.errstate(all='ignore'):
    cal_tab = pd.DataFrame({'AUPRC rete': [average_precision_score(lab_cal[ref_h[c], c], Pb[ref_h[c], c]) for c in CAL_OK],
                            'AUPRC rete + partner': [average_precision_score(lab_cal[ref_h[c], c], Pcal[ref_h[c], c]) for c in CAL_OK],
                            'ECE prima': [np.abs(pd.Series(y_c - p_c).groupby(np.minimum((p_c * 10).astype(int), 9)).sum()).sum() / len(p_c)
                                          for c in CAL_OK for y_c, p_c in [(lab_cal[ref_h[c], c], Pb[ref_h[c], c])]],
                            'ECE dopo': [np.abs(pd.Series(y_c - p_c).groupby(np.minimum((p_c * 10).astype(int), 9)).sum()).sum() / len(p_c)
                                         for c in CAL_OK for y_c, p_c in [(lab_cal[ref_h[c], c], Pcal[ref_h[c], c])]],
                            'con partner etichettati': [(NBN[ref_h[c], c] > 0).mean() for c in CAL_OK]},
                           index=np.array(ALL)[CAL_OK])
cal_tab['riferimenti (neg/pos)'] = [f'{n0}/{n1}' for n0, n1 in n_ref[CAL_OK]]
print(cal_tab.round(3).to_string())

# diagramma di affidabilità: frazione di veri positivi per intervallo di probabilità predetta
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, (titolo, cls) in zip(axes, [('Organelli', CAL_OK[CAL_OK < nC]), ('Neurone e mielina', CAL_OK[CAL_OK >= nC])]):
    for nome, P_, col in [('rete', Pb[:, :C], '#d62a2a'), ('rete + partner, calibrata', Pcal, '#2a78d6')]:
        p_all = np.concatenate([P_[ref_h[c], c] for c in cls] + [np.zeros(0)])
        y_all = np.concatenate([lab_cal[ref_h[c], c] for c in cls] + [np.zeros(0)])
        bins = np.minimum((p_all * 10).astype(int), 9)
        curva = pd.DataFrame({'p': p_all, 'y': y_all, 'b': bins}).groupby('b').mean()
        ax.plot(curva['p'], curva['y'], 'o-', color=col, lw=2, label=nome)
    ax.plot([0, 1], [0, 1], color='#c3c2b7', lw=1)
    ax.set(title=titolo, xlabel='Probabilità predetta')
    ax.legend(fontsize=9, frameon=False)
    ax.grid(color='#e6e5e0', lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Frazione osservata di positivi')
fig.suptitle('Calibrazione out-of-fold sulle proteine umane del cervello', fontsize=13)
fig.tight_layout()
fig.savefig(DATA / 'calibrazione.png', dpi=150)
plt.show()

## 4 · Fusione con le misure: sequenza + esperimenti che hanno già visto la proteina
Come nella fusione di sensori: la probabilità calibrata è il prior e le fonti che hanno misurato la proteina aggiungono
il loro rapporto di verosimiglianza (LLR). La versione precedente sommava i LLR di tutte le fonti con le sensibilità
stimate dal canale di rumore, e la revisione ha trovato tre difetti, corretti qui:

1. **Le fonti non sono indipendenti.** Le tre mappe LOPIT usano gli stessi marcatori e lo stesso classificatore, i due
   livelli di HPA sono lo stesso atlante. Ora le fonti dello stesso tipo contano **una volta**: nel gruppo si fa la
   media dei LLR, non la somma.
2. **Le sensibilità del canale erano troppo alte** (HPA Approved 0,46 stimata contro 0,18 osservata): un'assenza pesava
   troppo. Per i 9 organelli si usano sensibilità e specificità **osservate** contro le etichette curate (stimate fuori
   fold, con 20 pseudo-osservazioni della stima del canale per le classi con pochi dati). Per le classi neuronali, che
   non hanno etichette curate, resta la stima del canale.
3. **La calibrazione era misurata prima della fusione**, mentre la tabella finale mostra le probabilità dopo. Ora una
   logistica per classe, stimata fuori fold sulle proteine curate, impara quanto pesare prior e misure
   (`logit P = a·logit P_prior + b·LLR + c`): se le misure sono ridondanti `b` scende sotto 1. La calibrazione (ECE) e
   la quota di probabilità schiacciate a 0 o a 100 sono misurate **dopo** la fusione, fuori fold, contro le etichette
   DeepLoc, usando solo fonti indipendenti da DeepLoc (non UniProt).

Per le classi neuronali `b` non si può stimare (nessuna etichetta curata): si usa la media dei `b` degli organelli, e le
loro probabilità dopo la fusione restano **non verificate**. La tabella sotto confronta prior, fusione vecchia
(somma ingenua) e fusione rivista.


In [ ]:
chan_b = np.clip(z['chan'], 0.02, 0.995)                                  # [sens/spec, fonte, classe]: stima del canale di rumore
FUSE_ALL = ~np.isin(SOURCES, ['deeploc', 'uniprot_debole'])
FUSE_IND = FUSE_ALL & ~np.isin(SOURCES, ['uniprot_forte'])
lg_cal = np.log(np.clip(Pcal, 1e-4, 1 - 1e-4) / (1 - np.clip(Pcal, 1e-4, 1 - 1e-4)))
# la fusione della versione precedente, solo per confronto: somma dei LLR di tutte le fonti, sensibilità del canale
ing_pos, ing_neg = np.log(chan_b[0] / (1 - chan_b[1])), np.log((1 - chan_b[0]) / chan_b[1])
P_ingenua = 1 / (1 + np.exp(-(lg_cal + np.where(np.isnan(OBS[:, FUSE_IND, :C]), 0.0, np.where(OBS[:, FUSE_IND, :C] == 1, ing_pos[FUSE_IND],
                                                                                              ing_neg[FUSE_IND])).sum(1))))

# 1) sensibilità e specificità osservate contro le etichette curate, per i 9 organelli. Stimate fuori fold: per le proteine
#    della partizione k si usano solo le altre partizioni (l'ultima riga, k = −1, è per il test: tutte le proteine curate)
M_PSEUDO = 20
PARTI = [0, 1, 2, 3, 4, -1]
fi = pd.Series(np.arange(len(PARTI)), index=PARTI).reindex(part).fillna(len(PARTI) - 1).astype(int).to_numpy()
SENS, SPEC = np.repeat(chan_b[0][None], len(PARTI), 0), np.repeat(chan_b[1][None], len(PARTI), 0)       # [partizione, fonte, classe]
G9, O9 = OBS[:, :1, :nC], OBS[:, :, :nC]
for i_k, k in enumerate(PARTI):
    fuori = (part != k)[:, None, None]
    SENS[i_k, :, :nC] = (((O9 == 1) & (G9 == 1) & fuori).sum(0) + M_PSEUDO * chan_b[0][:, :nC]) / (((G9 == 1) & ~np.isnan(O9) & fuori).sum(0) + M_PSEUDO)
    SPEC[i_k, :, :nC] = (((O9 == 0) & (G9 == 0) & fuori).sum(0) + M_PSEUDO * chan_b[1][:, :nC]) / (((G9 == 0) & ~np.isnan(O9) & fuori).sum(0) + M_PSEUDO)
SENS, SPEC = np.clip(SENS, 0.02, 0.995), np.clip(SPEC, 0.02, 0.995)
llr_pos, llr_neg = np.log(SENS / (1 - SPEC)), np.log((1 - SENS) / SPEC)

# 2) le fonti dello stesso tipo contano una volta: media dei LLR nel gruppo, poi somma tra i gruppi
GRUPPI = {'lopit_2024': 'LOPIT', 'lopit_dc': 'LOPIT', 'lopit_hyper': 'LOPIT', 'hpa_alta': 'HPA', 'hpa_approved': 'HPA',
          'syngo': 'SynGO', 'syngo_funzionale': 'SynGO'}
W_g = pd.get_dummies(pd.Series(SOURCES).replace(GRUPPI)).to_numpy(float)              # [fonte, gruppo]
LLR = {}
for nome, O_, f_ in [('tutte', OBS_APP, FUSE_ALL), ('indipendenti', OBS, FUSE_IND)]:
    visto = (~np.isnan(O_[:, :, :C])) * f_[None, :, None]
    L_ = np.where(O_[:, :, :C] == 1, llr_pos[fi], llr_neg[fi]) * visto
    LLR[nome] = (np.einsum('nsc,sg->ngc', L_, W_g) / np.maximum(np.einsum('nsc,sg->ngc', visto, W_g), 1)).sum(1)

# 3) quanto pesare prior e misure: una logistica per classe sulle proteine curate, fuori fold. Si stima con le fonti
#    indipendenti da DeepLoc e si applica anche alla fusione con tutte le fonti, quella della tabella finale
is_hg = np.isin(np.arange(N_PROT), human_gold)
P_fus, P_fus_ind, pesi = Pcal.copy(), Pcal.copy(), []
for c in range(nC):
    for k in PARTI:
        fit_c, app_c = is_hg & (part != k), fi == PARTI.index(k)
        m_f = LogisticRegression(C=10.0, max_iter=1000).fit(np.c_[lg_cal[fit_c, c], LLR['indipendenti'][fit_c, c]], GOLD[fit_c, c])
        P_fus_ind[app_c, c] = m_f.predict_proba(np.c_[lg_cal[app_c, c], LLR['indipendenti'][app_c, c]])[:, 1]
        P_fus[app_c, c] = m_f.predict_proba(np.c_[lg_cal[app_c, c], LLR['tutte'][app_c, c]])[:, 1]
    pesi.append([m_f.coef_[0, 0], m_f.coef_[0, 1], m_f.intercept_[0]])                 # l'ultimo modello (k = −1) usa tutte le curate
pesi = pd.DataFrame(pesi, index=CLASSES, columns=['peso del prior (a)', 'peso delle misure (b)', 'intercetta (c)'])
# classi neuronali: nessuna etichetta curata per stimare i pesi. Si usa il peso medio delle misure negli organelli
B_NEURO = float(np.clip(pesi['peso delle misure (b)'].mean(), 0, 1))
P_fus[:, nC:] = 1 / (1 + np.exp(-(lg_cal[:, nC:] + B_NEURO * LLR['tutte'][:, nC:])))
P_fus_ind[:, nC:] = 1 / (1 + np.exp(-(lg_cal[:, nC:] + B_NEURO * LLR['indipendenti'][:, nC:])))
print('Pesi della fusione (b < 1: le misure sono in parte ridondanti tra loro o con il prior):')
print(pesi.round(2).to_string())

# verifica DOPO la fusione, fuori fold, sulle proteine curate misurate da almeno una fonte indipendente
meas = human_gold[(LLR['indipendenti'][human_gold, :nC] != 0).any(1)]
print(f'\nProteine DeepLoc umane misurate da fonti indipendenti: {len(meas):,}')
VERS = {'sequenza + partner (prior)': Pcal, 'fusione ingenua (versione precedente)': P_ingenua, 'fusione rivista': P_fus_ind}
with np.errstate(all='ignore'):
    fus_tab = pd.DataFrame({nome: [average_precision_score(GOLD[meas, c], P_[meas, c]) for c in range(nC)] for nome, P_ in VERS.items()}, index=CLASSES).T
    ver_tab = pd.DataFrame({'AUPRC macro': fus_tab.mean(axis=1),
                            'ECE medio': [np.mean([np.abs(pd.Series(GOLD[meas, c] - P_[meas, c]).groupby(np.minimum((P_[meas, c] * 10).astype(int), 9)).sum()).sum() / len(meas)
                                                   for c in range(nC)]) for P_ in VERS.values()],
                            'quota di probabilità < 1% o > 99%': [((P_[meas, :nC] < 0.01) | (P_[meas, :nC] > 0.99)).mean() for P_ in VERS.values()],
                            'errori sicuri (P < 1% su un positivo)': [((P_[meas, :nC] < 0.01) & (GOLD[meas] == 1)).sum() / max((GOLD[meas] == 1).sum(), 1)
                                                                      for P_ in VERS.values()]})
print('AUPRC per classe:')
print(fus_tab.round(3).to_string())
print('\nCalibrazione e saturazione dopo la fusione (fuori fold, contro le etichette DeepLoc):')
print(ver_tab.round(4).to_string())
ver_tab.round(4).to_excel(DATA / 'verifica_fusione.xlsx')


## 5 · Percentuali: quanta parte della proteina sta dove
Una probabilità di presenza non è una ripartizione. Le ripartizioni misurate su scala proteomica in cellule umane sono
le frazioni **citosolica** e **nucleare** (Hein 2024 in HEK293T, Itzhak 2016 in HeLa). Il secondo stadio unisce le
teste della rete, le probabilità calibrate, la frazione media dei vicini (CORUM e OpenCell separati, solo misure fuori
fold) e l'abbondanza. Il tetto è l'accordo tra i due esperimenti. ROC: "più di metà della proteina è nel citosol / nel
nucleo?".

In [ ]:
head = (1 / (1 + np.exp(-LOGIT[:, -2:]))).clip(1e-4, 1 - 1e-4)
stack = np.full((N_PROT, 2), np.nan)
FRAC_MODEL = {}                                                # come sopra: l'ultimo (k = −1) serve per le proteine nuove
for k in [0, 1, 2, 3, 4, -1]:
    NB = np.full((N_PROT, 2, 2), np.nan)                                     # [proteina, grafo, frazione]
    for g, tipo in enumerate(['corum', 'opencell']):
        e = grafo[(grafo['tipo'] == tipo).to_numpy() & (part[grafo['pid_a'].to_numpy()] == k)]
        lab_e = np.where((part[e['pid_b'].to_numpy()] != k)[:, None], FRAC[e['pid_b'].to_numpy()], np.nan)
        nb_mean = pd.DataFrame(lab_e, index=e['pid_a'].to_numpy()).groupby(level=0).mean()
        NB[nb_mean.index, g] = nb_mean.to_numpy()
    STK = np.c_[np.log(head / (1 - head)), lg_cal[:, :nC], NB.reshape(N_PROT, -1), proteins['ctx_log_copie_hek'].to_numpy()]
    for t in range(2):
        fit_k, app_k = oof_ok & np.isfinite(FRAC[:, t]) & (part != k), has_pred & (part == k)
        hgb_f = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=7, min_samples_leaf=30,
                                              l2_regularization=1.0, random_state=SEED)
        # una colonna senza nessun valore nelle righe di stima (es. nessun vicino misurato in quel fold) manda in errore
        # la binarizzazione di scikit-learn: diventa 0, cioè "nessuna informazione"
        X_k = np.where(np.isnan(STK[fit_k]).all(0), 0.0, STK)
        stack[app_k, t] = hgb_f.fit(X_k[fit_k], FRAC[fit_k, t]).predict(X_k[app_k]).clip(0, 1)
        FRAC_MODEL[t] = (hgb_f, np.isnan(STK[fit_k]).all(0))

is_mito = np.nan_to_num(OBS[:, np.isin(SOURCES, ['deeploc', 'uniprot_forte', 'hpa_alta', 'mitocarta']), CLASSES.index('Mitochondrion')]).max(1) == 1
rows_f = []
for t, (nome, c_hek, c_hela) in enumerate([('citosolica', 'frac_cyto_hek', 'frac_cyto_hela'), ('nucleare', 'frac_nuc_hek', 'frac_nuc_hela')]):
    fm = oof_ok & np.isfinite(FRAC[:, t]) & ~(is_mito & (t == 1))                # il nucleo si valuta fuori dai mitocondri
    both = proteins.loc[fm, [c_hek, c_hela]].dropna()
    for metodo, v in [('quota normalizzata', Pb[:, t] / Pb[:, :nC].sum(1)), ('testa della rete', head[:, t]),
                      ('testa + contesto molecolare', stack[:, t])]:
        rows_f.append({'frazione': nome, 'metodo': metodo, 'proteine': fm.sum(),
                       'Spearman': spearmanr(v[fm], FRAC[fm, t])[0], 'MAE': np.abs(v[fm] - FRAC[fm, t]).mean()})
    rows_f.append({'frazione': nome, 'metodo': 'tetto: HEK293T vs HeLa', 'proteine': len(both),
                   'Spearman': spearmanr(both[c_hek], both[c_hela])[0], 'MAE': (both[c_hek] - both[c_hela]).abs().mean()})
frac_tab = pd.DataFrame(rows_f).set_index(['frazione', 'metodo'])
print('Frazioni misurate, valutazione out-of-fold:')
print(frac_tab.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
for t, (ax, nome) in enumerate(zip(axes, ['citosol solubile', 'nucleo'])):
    fm = oof_ok & np.isfinite(FRAC[:, t]) & ~(is_mito & (t == 1))
    y_b = (FRAC[fm, t] >= 0.5).astype(int)
    for metodo, v, col, ls in [('quota normalizzata (ingenua)', Pb[:, t] / Pb[:, :nC].sum(1), '#d62a2a', '--'),
                               ('testa della rete', head[:, t], '#2a78d6', '-.'), ('testa + contesto molecolare', stack[:, t], '#eb6834', '-')]:
        fpr, tpr, _ = roc_curve(y_b, v[fm])
        ax.plot(fpr, tpr, color=col, ls=ls, lw=2, label=f'{metodo} (AUC {roc_auc_score(y_b, v[fm]):.3f})')
    ax.plot([0, 1], [0, 1], color='#c3c2b7', lw=1)
    ax.set(title=f'≥ 50% nel {nome} ({y_b.sum():,} su {len(y_b):,} proteine misurate)', xlabel='Falsi positivi (FPR)')
    ax.legend(fontsize=8, loc='lower right', frameon=False)
    ax.grid(color='#e6e5e0', lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Veri positivi (TPR)')
fig.suptitle('Percentuali misurate (HEK293T, HeLa): predizioni out-of-fold', fontsize=13)
fig.tight_layout()
fig.savefig(DATA / 'roc_frazioni.png', dpi=150)
plt.show()

## 6 · Tabella finale: proteine umane del cervello, senza sangue
Per ogni proteina l'uscita principale è **"% probabilità"** per ognuno dei 14 compartimenti: probabilità tarata, unita
ai partner e alle misure esistenti (fusione verificata nella sezione 4 per gli organelli; per i compartimenti neuronali
la probabilità è un ordinamento, non una frequenza, e la mielina non è valutabile). Non somma a 100 (una proteina può stare in più posti); i compartimenti neuronali
sono vuoti fuori dai neuroni, la mielina fuori dagli oligodendrociti. Accanto:
- **localizzazioni (P ≥ 50%)**, sempre almeno una (la più probabile tra gli organelli);
- **predette dalla sola sequenza** (soglie Jaccard dello 02, senza misure) e **insieme ad alta sensibilità** (le classi
  sopra la soglia che tiene il 90% dei positivi curati fuori fold; sul test HPA dello 02 il richiamo misurato è più
  basso, circa 0,81–0,85: non è una garanzia conformale);
- **ripartizione %** (somma 100): citosol solubile e nucleo misurati quando possibile, altrimenti predetti; il resto
  (pool sedimentabile: organelli, citoscheletro, grandi complessi) si ripartisce secondo la presenza;
- le **misure** che esistono (fonte: classi), il contesto cerebrale (espressione in regioni, neuroni, glia,
  oligodendrociti; specificità) e la `card_protein`.

In [ ]:
# soglie dalla sola sequenza (stesso metodo dello 02, sulle predizioni out-of-fold delle proteine oro)
grid_t, SCALES, EPS = np.linspace(0.02, 0.95, 94), np.linspace(0.8, 3.2, 49), 0.10
Yg, Pg = GOLD[gold_rows].astype(bool), Pb[gold_rows, :nC]
pr = Pg[:, :, None] >= grid_t[None, None]
tp, fp = (pr & Yg[:, :, None]).sum(0).astype(float), (pr & ~Yg[:, :, None]).sum(0).astype(float)
fn, tn = Yg.sum(0)[:, None] - tp, (~Yg).sum(0)[:, None] - fp
with np.errstate(all='ignore'):
    thr_b = grid_t[np.nan_to_num((tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))).argmax(1)]
pred_s = (Pg[None] >= (thr_b[None] * SCALES[:, None]).clip(0, .99)[:, None]) | (Pg == Pg.max(1, keepdims=True))[None]
thr_b = (thr_b * SCALES[int(np.argmax(((pred_s & Yg[None]).sum(2) / (pred_s | Yg[None]).sum(2)).mean(1)))]).clip(0, .99)
conf_b = np.array([np.quantile(np.r_[Pb[human_gold, c][GOLD[human_gold, c] == 1], 0.0], EPS) for c in range(nC)])

est = np.where(np.isfinite(FRAC), FRAC, stack)
est = est / np.maximum(1, est.sum(1, keepdims=True))
rest_cls = [0] + list(range(2, nC))
others = P_fus[:, rest_cls] / (P_fus[:, rest_cls].sum(1, keepdims=True) + 1e-9)
PERC = 100 * (1 - est.sum(1, keepdims=True)) * others
PERC = np.insert(PERC, 1, 100 * est[:, 1], axis=1)
PERC[:, 0] += 100 * est[:, 0]

bmask = has_pred & is_brain & is_human & ~(is_blood & ESCLUDI_SANGUE)
P_tab = np.where(gate, P_fus, np.nan)
top_org = P_fus[:, :nC] == P_fus[:, :nC].max(1, keepdims=True)
loc50 = np.c_[(P_fus[:, :nC] >= 0.5) | top_org, np.nan_to_num(P_tab[:, nC:]) >= 0.5]
misure = pd.Series('', index=np.flatnonzero(bmask))
for s_i, s_n in enumerate(SOURCES):
    lista = pd.DataFrame(OBS_APP[bmask, s_i, :] == 1, columns=ALL).dot(pd.Index(ALL) + ', ').str.rstrip(', ').to_numpy()
    misure = misure + np.where(lista != '', s_n + ': ' + lista + '; ', '')
table = proteins.loc[bmask, ['pid', 'uniprot_id', 'gene', 'protein_name', 'source', 'is_neuronal', 'is_oligo']].copy()
table[[f'% probabilità {c}' for c in ALL]] = (100 * P_tab[bmask]).round(1)
table['localizzazioni (P ≥ 50%)'] = ['; '.join(np.array(ALL)[m]) for m in loc50[bmask]]
table['predette dalla sola sequenza'] = ['; '.join(np.array(CLASSES)[np.maximum(p >= thr_b, p == p.max())]) for p in Pb[bmask, :nC]]
table['insieme ad alta sensibilità'] = ['; '.join(np.array(CLASSES)[p >= conf_b]) for p in Pb[bmask, :nC]]
table[[f'ripartizione % {c}' for c in CLASSES]] = PERC[bmask].round(1)
table['origine frazioni'] = np.where(np.isfinite(FRAC[bmask, 0]), 'misurata (HEK293T, HeLa)', 'predetta')
table['misure (fonte: classi)'] = misure.str.rstrip('; ').to_numpy()
table['fonti che la misurano'] = (~np.isnan(OBS_APP[bmask][:, FUSE_ALL, :C]).all(2)).sum(1)
table['partner con etichetta'] = NBN[bmask].max(1).astype(int)
for c_ in ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm', 'oligo_ncpm', 'brain_specificity', 'secreto_cervello']:
    table[c_] = proteins.loc[bmask, c_].to_numpy()
table['card_protein'] = proteins.loc[bmask, 'card_protein'].to_numpy()
print(f'Proteine del cervello in tabella: {bmask.sum():,} | escluse perché del sangue: {(has_pred & is_brain & is_blood).sum():,}')
print(table.drop(columns=['card_protein', 'misure (fonte: classi)']).head(3).T.to_string())
table.to_excel(DATA / 'probabilita_proteine_cervello.xlsx', index=False)
table.to_parquet(DATA / 'probabilita_proteine_cervello.parquet', index=False)

## 6b · Proteine nuove: una sequenza che non è nei dati, gli stessi ragionamenti
Per analizzare una proteina che non è tra le ~15.000 del progetto (una isoforma, una proteina di un'altra lista, una
costruzione sperimentale) basta scriverla in un file FASTA in `data/esterni/nuove/` e rilanciare il notebook:
```
>NOME_LIBERO  P12345
MSEQUENZA...
```
L'accession UniProt nell'intestazione è facoltativa: se c'è, la scheda testuale (card_protein) si costruisce da UniProt
con la stessa pulizia del notebook 01 e la struttura si prende da AlphaFold. Senza accession la scheda è vuota e la
rete lavora sulla sola sequenza (il dropout di blocco l'ha allenata anche a questo). Una scheda scritta a mano si può
mettere in `data/esterni/nuove/schede.csv` (colonne `nome`, `card`), in inglese e senza parole di localizzazione.

Passaggi, identici a quelli delle proteine del progetto:
1. embedding ESM-2 e ProtT5 con gli stessi 8 riassunti (proteina intera, inizio, centro, fine);
2. descrittori di sorting e punteggi dei motivi terminali (le matrici dei 5 fold dello 02);
3. scheda testuale → PubMedBERT; struttura → descrittori AlphaFold;
4. i 5 modelli dello stadio `cervello` (media dei logit, come per il test HPA);
5. calibrazione con il modello stimato su tutte le proteine (senza partner: una proteina nuova non è nel grafo);
6. percentuali con il secondo stadio delle frazioni; insieme ad alta sensibilità e soglie della sola sequenza;
7. le proteine del progetto più simili (embedding medi) con le loro card_protein e localizzazioni note.

I compartimenti neuronali si mostrano sempre, perché l'espressione nei neuroni di una proteina nuova non è nota: vanno
letti come "se la proteina è espressa nei neuroni".

In [ ]:
# 6b · proteine nuove dal FASTA in data/esterni/nuove/
import io

import requests
import torch
import torch.nn as nn
from scipy.spatial.distance import cdist
from transformers import AutoModel, AutoTokenizer, EsmModel, T5EncoderModel, T5Tokenizer

NUOVE_DIR = DATA / 'esterni' / 'nuove'
NUOVE_DIR.mkdir(parents=True, exist_ok=True)
ACC_RE = r'\b(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})\b'
voci = re.findall(r'>([^\n]*)\n([^>]*)', ''.join(f.read_text(encoding='utf-8', errors='ignore') + '\n' for f in sorted(NUOVE_DIR.glob('*.fa*'))))
new = pd.DataFrame({'nome': [(h.split() + ['senza_nome'])[0] for h, _ in voci], 'acc': [(re.findall(ACC_RE, h) + [''])[0] for h, _ in voci],
                    'sequence': [re.sub(r'[^A-Z]', '', q.upper()) for _, q in voci]}, columns=['nome', 'acc', 'sequence']).astype(str)
new = new[new['sequence'].str.len() >= 20].drop_duplicates('nome').reset_index(drop=True)
schede = pd.concat([pd.read_csv(f, encoding='utf-8') for f in sorted(NUOVE_DIR.glob('schede.csv'))] + [pd.DataFrame(columns=['nome', 'card'])], ignore_index=True)
N_NEW = len(new)
D_UNIT = np.load(DATA / 'unit.npy', mmap_mode='r').shape[1]
LOG_NEW, UNIT_NEW, AUX_NEW = np.zeros((0, C + A + 2), np.float32), np.zeros((0, D_UNIT), np.float32), np.zeros((0, A), np.float32)
MECC_LAB = {'aux_peptide_segnale': 'peptide segnale', 'aux_transit_peptide': 'presequenza mitocondriale', 'aux_tm_1piu': "almeno un'elica TM",
            'aux_tm_2piu': 'più eliche TM', 'aux_ancora_lipidica': 'ancora lipidica/GPI', 'aux_glicoproteina': 'glicosilazione',
            'aux_nls': 'segnale di localizzazione nucleare', 'aux_palmitoilazione': 'palmitoilazione',
            'aux_via_secretoria': 'via secretoria', 'aux_nucleo_citoplasma': 'nucleo-citoplasma', 'aux_import_organelli': 'import negli organelli'}
table_new = pd.DataFrame(columns=table.columns)
print(f'Proteine nuove trovate in {NUOVE_DIR}: {N_NEW} {list(new["nome"])}')

# scheda testuale da UniProt (stessi campi e stessa pulizia del notebook 01; poi la seconda pulizia dello 02)
UP_FIELDS = ('accession,gene_names,protein_name,protein_families,cc_function,cc_catalytic_activity,cc_pathway,ft_domain,cc_domain,'
             'cc_subunit,cc_ptm,go_p,go_f,keyword,cc_tissue_specificity,cc_disease')
up_new = []
for a_ in new.loc[new['acc'] != '', 'acc']:
    try:
        r_ = requests.get('https://rest.uniprot.org/uniprotkb/stream', timeout=60,
                          params={'query': f'accession:{a_}', 'fields': UP_FIELDS, 'format': 'tsv'})
        up_new.append(pd.read_csv(io.StringIO(r_.text), sep='\t', dtype=str))
    except Exception as err:
        print(f'UniProt non raggiungibile per {a_} ({type(err).__name__}): scheda vuota')
upn = pd.concat(up_new + [pd.DataFrame(columns=['Entry'])], ignore_index=True).rename(columns={
    'Entry': 'acc', 'Gene Names': 'gene_names', 'Protein names': 'protein_name', 'Protein families': 'family', 'Function [CC]': 'function',
    'Catalytic activity': 'catalytic', 'Pathway': 'pathway', 'Domain [FT]': 'domain_ft', 'Domain [CC]': 'domain_cc',
    'Subunit structure': 'subunit', 'Post-translational modification': 'ptm', 'Gene Ontology (biological process)': 'go_bp',
    'Gene Ontology (molecular function)': 'go_mf', 'Keywords': 'keywords', 'Tissue specificity': 'tissue', 'Involvement in disease': 'disease'})
TXT = ['protein_name', 'function', 'catalytic', 'pathway', 'subunit', 'domain_cc', 'ptm', 'tissue', 'disease', 'family', 'go_bp', 'go_mf', 'keywords']
src_n = (new[['nome', 'acc']].merge(upn.drop_duplicates('acc'), on='acc', how='left').set_index('nome')
         .reindex(columns=['acc', 'gene_names', 'domain_ft'] + TXT).astype(object))   # i campi mancanti restano vuoti
clean = src_n.reindex(columns=TXT).fillna('').astype(str).replace(
    {r'\{ECO:[^}]*\}': '', r'\(PubMed:[^)]*\)': '', r'\[MIM:\d+\]': '', r'\[GO:\d+\]': '', r'Xref=[^;]*;?': '',
     r'Evidence=[^;]*;?': '', r'PhysiologicalDirection=[^;]*;?': '', r'Note=': '', r'\b[A-Z][A-Z ]{2,}: ': ' '}, regex=True)
LOC_WORDS = (r'(?i)\b(?:nucle\w*|nucleol\w*|mitochondri\w*|cytoplasm\w*|cytosol\w*|membran\w*|endoplasmic|reticul\w*|'
             r'golgi\w*|lysosom\w*|vacuol\w*|peroxisom\w*|secret\w*|extracellular|periplasm\w*|endosom\w*|vesic\w*|'
             r'synap\w*|axon\w*|dendrit\w*|myelin\w*|organell\w*|lumen\w*|centrosom\w*|centriol\w*|chromosom\w*|'
             r'chromatin|cytoskelet\w*|microtubul\w*|plasma|cell surface|apical|basolateral|perinuclear|sarcoplasm\w*|'
             r'sarcolemm\w*|kinetochore|envelope|intermembrane|matri(?:x|ces)|cili(?:um|a|ary)|flagell\w*|spindle|'
             r'signal peptide|signal sequence|transit peptide|transmembrane\w*|anchor\w*|lipoprotein|gpi|translocat\w*|'
             r'import\w*|export\w*|shuttl\w*|targeting|sorting|retention|retriev\w*|compartment\w*|subcellular|'
             r'localiz\w*|located|location|exocyt\w*|endocyt\w*|neurite|growth cone|bouton|postsynaptic|presynaptic|'
             r'nuclear|exosom\w*|lipid droplet|granule|p-body|cajal|speckle|histone|ribosom\w*|outer|inner)\b')
gene_n = src_n['gene_names'].fillna('').astype(str).str.split().str[0].fillna('').astype(str).str.upper()
gene_n = gene_n.where(gene_n != '', pd.Series(new['nome'].to_numpy(), index=src_n.index))
parts = [('Protein: ', clean['protein_name']), ('Gene: ', gene_n), ('Family: ', clean['family']), ('Function: ', clean['function']),
         ('Catalytic activity: ', clean['catalytic']), ('Pathway: ', clean['pathway']),
         ('Domains: ', src_n['domain_ft'].fillna('').astype(str).str.findall(r'/note="([^"]+)"').str.join(', ')),
         ('Domain notes: ', clean['domain_cc']), ('Subunit and partners: ', clean['subunit']), ('Post-translational modifications: ', clean['ptm']),
         ('Biological processes: ', clean['go_bp'].str.split(r'\s*;\s*').str[:25].str.join('; ')),
         ('Molecular functions: ', clean['go_mf'].str.split(r'\s*;\s*').str[:15].str.join('; ')),
         ('Keywords: ', clean['keywords']), ('Tissue specificity: ', clean['tissue']), ('Disease: ', clean['disease'])]
card_n = pd.Series('', index=src_n.index, dtype=object)
for label, field in parts:
    f_ = field.fillna('').astype(str).str.strip().str.rstrip('.;')
    card_n = card_n + np.where(f_.str.len() > 0, label + f_ + '. ', '')
card_n = (card_n.str.replace(LOC_WORDS, '', regex=True).str.replace(r'\(\s*\)|\[\s*\]', '', regex=True)
          .str.replace(r'(?i)\b(?:in|of|to|into|at|on|from|within|the|a|an)(?:\s+(?:the|a|an))?\s*(?=[.,;:])', '', regex=True)
          .str.replace(r'\s+', ' ', regex=True).str.replace(r'\s+([.,;:])', r'\1', regex=True)
          .str.replace(r'[;,]\s*\.', '.', regex=True).str.replace(r'\.(\s*\.)+', '.', regex=True))
card_n = (card_n + 'Length: ' + new['sequence'].str.len().astype(str).to_numpy() + ' amino acids.').str.replace(cfg['SCRUB2'], '', regex=True)
card_n = card_n.where(src_n['acc'].fillna('') != '', 'Unknown protein. Length: ' + new['sequence'].str.len().astype(str).to_numpy() + ' amino acids.')
card_n.update(schede.drop_duplicates('nome').set_index('nome')['card'])
card_n = card_n.str.slice(0, 6000)

for _ in range(int(N_NEW > 0)):
    DEV = ['cpu', 'cuda'][int(torch.cuda.is_available())]
    MAXR = cfg['MAX_RES']
    sq = new['sequence']
    trunc_n = sq.where(sq.str.len() <= MAXR, sq.str[:MAXR // 2] + sq.str[-(MAXR // 2):])

    # 1. embedding: stessi 8 riassunti dello 02 (il layer intermedio è hidden_states[MID + 1], l'uscita del blocco MID)
    tok_e, m_e = AutoTokenizer.from_pretrained(cfg['ESM']), EsmModel.from_pretrained(cfg['ESM']).to(DEV).eval()
    MID = round(m_e.config.num_hidden_layers * 2 / 3)
    E_N = []
    for s_ in trunc_n:
        with torch.no_grad():
            o_ = m_e(**tok_e([s_], return_tensors='pt').to(DEV), output_hidden_states=True)
        L_, r_ = len(s_), np.arange(len(s_))
        h_l, h_m = o_.last_hidden_state[0, 1:L_ + 1].float().cpu().numpy(), o_.hidden_states[MID + 1][0, 1:L_ + 1].float().cpu().numpy()
        E_N.append(np.concatenate([h_l.mean(0), h_m.mean(0), h_l.max(0)] +
                                  [h_l[m_].mean(0) for m_ in [r_ < 20, r_ < 60, (r_ >= L_ / 3) & (r_ < 2 * L_ / 3), r_ >= L_ - 15, r_ >= L_ - 50]]))
    del m_e
    E5_N = []
    for _t5 in range(cfg['USE_T5']):
        tok_5, m_5 = T5Tokenizer.from_pretrained(cfg['T5'], do_lower_case=False), T5EncoderModel.from_pretrained(cfg['T5']).to(DEV).eval()
        MID5 = round(m_5.config.num_layers * 2 / 3)
        for s_ in trunc_n:
            with torch.no_grad():
                o_ = m_5(**tok_5([' '.join(re.sub('[UZOB]', 'X', s_))], return_tensors='pt').to(DEV), output_hidden_states=True)
            L_, r_ = len(s_), np.arange(len(s_))
            h_l, h_m = o_.last_hidden_state[0, :L_].float().cpu().numpy(), o_.hidden_states[MID5 + 1][0, :L_].float().cpu().numpy()
            E5_N.append(np.concatenate([h_l.mean(0), h_m.mean(0), h_l.max(0)] +
                                       [h_l[m_].mean(0) for m_ in [r_ < 20, r_ < 60, (r_ >= L_ / 3) & (r_ < 2 * L_ / 3), r_ >= L_ - 15, r_ >= L_ - 50]]))
        del m_5
    E_N = np.array(E_N, np.float16).astype(np.float32)                    # stessa precisione degli embedding salvati
    E5_N = np.array(E5_N, np.float16).astype(np.float32).reshape(N_NEW, -1)
    D_, D5_, SUMS = cfg['D'], cfg['D5'], cfg['SUMMARIES']
    BN = {f'esm_{s}': E_N[:, i * D_:(i + 1) * D_] for i, s in enumerate(SUMS)}
    BN.update({f't5_{s}': E5_N[:, i * D5_:(i + 1) * D5_] for i, s in enumerate(SUMS[:len(SUMS) * cfg['USE_T5']])})

    # 2. descrittori di sorting: stesse formule della sezione 3b dello 02
    AA = 'ACDEFGHIKLMNPQRSTVWY'
    KD = dict(zip(AA, [1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8, 1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3]))
    EIS = dict(zip(AA, [0.62, 0.29, -0.90, -0.74, 1.19, 0.48, -0.40, 1.38, -1.50, 1.06, 0.64, -0.78, 0.12, -0.85, -2.53, -0.18, -0.05, 1.08, 0.81, 0.26]))
    n_len, nt, ct = sq.str.len(), sq.str[:30], sq.str[-30:]
    hand_n = pd.DataFrame({f'aa_{a}': sq.str.count(a) / n_len for a in AA})
    hand_n['log_len'] = np.log(n_len)
    hand_n['nt_kd'] = sum(nt.str.count(a) * KD[a] for a in AA) / nt.str.len()
    hand_n['nt_charge'] = (nt.str.count('[KR]') - nt.str.count('[DE]')) / 30
    hand_n['nt_hydro_core'] = nt.str.contains(r'[AILMFVWC]{7,}').astype(int)
    hand_n['tm_like'] = sq.str.count(r'[^DEKRHNQPS]{17,}')
    hand_n['nls'] = sq.str.count(r'K[KR].[KR]') + sq.str.count(r'[KR]{2}.{10,12}[KR]{3}')
    hand_n['kdel'] = sq.str.contains(r'[KRHQSA][DENQ]EL$').astype(int)
    hand_n['kkxx'] = sq.str.contains(r'K.K..$|KK..$').astype(int)
    hand_n['pts1'] = sq.str.contains(r'[SAC][KRH][LM]$').astype(int)
    hand_n['caax'] = sq.str.contains(r'C[AILMV][AILMV].$').astype(int)
    hand_n['myr'] = sq.str.contains(r'^MG[^EDRKHPFYW]..[STAGCN][^P]').astype(int)
    hand_n['ct_kd'] = sum(ct.str.count(a) * KD[a] for a in AA) / ct.str.len()
    helix = np.exp(1j * np.deg2rad(100) * np.arange(18))
    sort_n = []
    for s in sq:
        n = len(s)
        win = np.convolve(np.array([KD.get(a, 0.0) for a in s]), np.ones(19) / 19, 'valid')
        covered = np.zeros(n, bool)
        for i0 in np.flatnonzero(win >= 1.6):
            covered[i0:i0 + 19] = True
        seg = list(re.finditer('1+', ''.join(np.where(covered, '1', '0'))))
        runs, starts = [m.end() - m.start() for m in seg], [m.start() for m in seg]
        s0, r0 = (starts + [n])[0], (runs + [0])[0]
        eis = np.array([EIS.get(a, 0.0) for a in s[:40]])
        sort_n.append({'nt_muH': max(abs((eis[i0:i0 + 18] * helix[:len(eis[i0:i0 + 18])]).sum()) / 18 for i0 in range(max(1, len(eis) - 17))),
                       'nt_acid20': sum(a in 'DE' for a in s[:20]), 'nt_R20': s[:20].count('R'),
                       'tm_n': len(runs), 'tm_maxlen': max(runs + [0]), 'tm_meanlen': sum(runs) / max(len(runs), 1),
                       'tm_first_start': s0 / n, 'tail_anchor': int(len(runs) == 1 and n - (s0 + r0) < 30),
                       'signal_anchor': int(len(runs) >= 1 and s0 < 50),
                       'pos_inside': sum(a in 'KR' for a in s[max(0, s0 - 15):s0]) - sum(a in 'KR' for a in s[s0 + r0:s0 + r0 + 15])})
    sort_n = pd.DataFrame(sort_n)
    ct40, nt40 = sq.str[-40:], sq.str[:40]
    sort_n['yxxphi_ct'] = ct40.str.count(r'Y..[LIMFV]')
    sort_n['dileu_ct'] = ct40.str.count(r'[DE]...L[LI]')
    sort_n['diacid_ct'] = ct40.str.count(r'D.E')
    sort_n['ff_ct'] = ct40.str.count('FF')
    sort_n['pts2'] = nt40.str.contains(r'[RK][LVIQ]..[LVIHQ][LSGAK].[HQ][LAF]').astype(int)
    sort_n['pts1_ext'] = sq.str.contains(r'[SACKNP][KRH][LMI]$').astype(int)
    sort_n['nes'] = sq.str.count(r'[LIVFM].{2,3}[LIVFM].{2,3}[LIVFM].[LIVFM]') / n_len * 100
    sort_n['pdz_I'] = sq.str.contains(r'[ST].[VIL]$').astype(int)
    sort_n['pdz_II'] = sq.str.contains(r'[VLIFY].[VLIFY]$').astype(int)
    sort_n['nt_cys_palm'] = nt.str[:20].str.count('C')
    sort_n['rg_rich'] = sq.str.count('RG|GR') / n_len * 100
    sort_n['glyc_sequon'] = sq.str.count(r'N[^P][ST]') / n_len * 100
    sort_n['idr_like'] = sq.str.count('[PESQK]') / n_len
    sort_n['ankG_bind'] = sq.str.contains(r'V[PA][IL]A..E[SD]D').astype(int)
    hand_n = pd.concat([hand_n, sort_n.set_index(hand_n.index)], axis=1).reindex(columns=cfg['HAND_COLS']).astype(np.float16).astype(np.float32)
    BN['hand'] = hand_n.to_numpy()

    # 3. scheda testuale con PubMedBERT (blocchi da 512 token, media dei blocchi)
    tok_t, bert = AutoTokenizer.from_pretrained(cfg['TEXT_MODEL']), AutoModel.from_pretrained(cfg['TEXT_MODEL']).to(DEV).eval()
    CARD_N = []
    for t_ in card_n.reindex(new['nome']).fillna('Unknown protein.'):
        enc = tok_t([t_], truncation=True, max_length=512, stride=64, return_overflowing_tokens=True, padding=True, return_tensors='pt')
        enc.pop('overflow_to_sample_mapping')
        with torch.no_grad():
            m_att = enc['attention_mask'].to(DEV).unsqueeze(-1).float()
            h = bert(input_ids=enc['input_ids'].to(DEV), attention_mask=enc['attention_mask'].to(DEV)).last_hidden_state.float()
        CARD_N.append(((h * m_att).sum(1) / m_att.sum(1)).mean(0).cpu().numpy())
    del bert
    BN['card'] = np.array(CARD_N, np.float16).astype(np.float32)

    #    struttura: modello AlphaFold v6 dell'accession, se c'è; stessi 15 descrittori della cella 2d
    AFD = Path(cfg['AF_DIR'])
    CA5 = re.compile(r'^ATOM\s+\d+\s+C\s+CA\s+\S+\s+\S+\s+\S+\s+\S+\s+(\d+)\s+\S+\s+(\S+)\s+(\S+)\s+(\S+)\s+\S+\s+(\S+)', re.M)
    ST_N = np.zeros((N_NEW, len(cfg['STRUCT'])), np.float32)
    for i, a_ in enumerate(new['acc']):
        f_ = AFD / f'{a_}.cif'
        for _d in range(int(a_ != '' and not (f_.exists() and f_.stat().st_size > 0))):
            try:
                r_ = requests.get(f'https://alphafold.ebi.ac.uk/files/AF-{a_}-F1-model_v6.cif', timeout=60)
                f_.write_bytes(r_.content * int(r_.status_code == 200))
            except Exception as err:
                print(f'AlphaFold non raggiungibile per {a_} ({type(err).__name__})')
        ca_ = np.array(CA5.findall(''.join(p_.read_text(errors='ignore') for p_ in compress([f_], [f_.exists()]))), float).reshape(-1, 5)
        for _s in range(int(len(ca_) >= 10)):
            xyz, b = ca_[:, 1:4], ca_[:, 4]
            n, c_ = len(b), ca_[:, 1:4] - ca_[:, 1:4].mean(0)
            ev = np.sort(np.linalg.eigvalsh(c_.T @ c_ / n))
            dist, sep = cdist(c_, c_), np.abs(np.arange(n)[:, None] - np.arange(n)[None])
            cont = (dist < 8.0) & (sep >= 3)
            lungo = sep[np.triu_indices(n, 1)][cont[np.triu_indices(n, 1)]]
            basso = b < 50
            bordi = np.flatnonzero(np.diff(np.r_[0, basso.astype(int), 0]))
            ST_N[i] = [b.mean(), b.std(), basso.mean(), (b >= 90).mean(), b[:30].mean(), b[-30:].mean(), np.sqrt(ev.sum()) / n ** 0.4,
                       1.5 * (ev ** 2).sum() / ev.sum() ** 2 - 0.5, 1 - 3 * (ev[0] * ev[1] + ev[0] * ev[2] + ev[1] * ev[2]) / (ev.sum() ** 2 + 1e-12),
                       cont.sum() / 2 / n, lungo.sum() / max(len(lungo), 1) / n, (bordi[1::2] - bordi[::2]).max(initial=0) / n,
                       basso[:30].mean(), basso[-30:].mean(), 1.0]
    BN['struct'] = ST_N.astype(np.float16).astype(np.float32)

    # motivi terminali: per il fold k le matrici stimate fuori da k (come per il test HPA)
    PWM = np.load(DATA / 'pwm_motivi.npz')
    OH = [np.stack([(s_.str[j].to_numpy()[:, None] == np.array(list(AA))[None]) for j in range(n_)], 1).astype(np.float32)
          for n_, s_ in [(12, sq.str[-12:].str.rjust(12, 'X')), (30, sq.str[:30].str.ljust(30, 'X'))]]
    for k in cfg['FOLDS']:
        BN[f'motivi@{k}'] = np.hstack([np.einsum('njl,cjl->nc', OH[t], PWM[f'k{k}_j-1_t{t}']) for t in range(2)]).astype(np.float16).astype(np.float32)

    # 4. i 5 modelli dello stadio "cervello", ricostruiti con la stessa architettura; media dei logit
    logits = []
    for k in cfg['FOLDS']:
        ck = torch.load(DATA / 'modello_cervello' / f'fold{k}.pt', map_location='cpu', weights_only=False)
        run, spans, T = ck['run'], ck['spans'], int(ck['run']['tok'])
        proj = nn.ModuleList([nn.Sequential(nn.Dropout(run['in_drop']), nn.Linear(b1 - b0, T), nn.LayerNorm(T)) for b0, b1 in spans])
        attn = [nn.TransformerEncoder(nn.TransformerEncoderLayer(T, 4, 2 * T, run['dropout'], activation='gelu', batch_first=True, norm_first=True),
                                      max(1, int(run['fusione'])), enable_nested_tensor=False)] * int(run['fusione'] > 0)
        hidden = [nn.GELU(), *attn, nn.Flatten(1), nn.Dropout(run['dropout']), nn.Linear(len(spans) * T, run['hidden']),
                  nn.LayerNorm(run['hidden']), nn.GELU(), nn.Dropout(run['dropout'])]
        hidden += sum([[nn.Linear(run['hidden'], run['hidden']), nn.LayerNorm(run['hidden']), nn.GELU(), nn.Dropout(run['dropout'])]
                       for _ in range(run['depth'])], [])
        net = nn.ModuleDict({'proj': proj, 'body': nn.Sequential(*hidden, nn.Linear(run['hidden'], ck['C'] + ck['A'] + 2))})
        net.load_state_dict(ck['state'])
        net.eval()
        x_ = torch.tensor(np.hstack([BN[{'motivi': f'motivi@{k}'}.get(b, b)] for b in run['blocks']]), dtype=torch.float32)
        x_ = ((x_ - ck['mu']) / ck['sd']).clamp(-20, 20)
        with torch.no_grad():
            logits.append(net['body'](torch.stack([pj(x_[:, b0:b1]) for pj, (b0, b1) in zip(net['proj'], spans)], 1)).numpy())
    LOG_NEW = np.mean(logits, 0)

    # 5–6. calibrazione (senza partner), frazioni, percentuali, soglie: stessi modelli delle proteine del progetto
    P_new = 1 / (1 + np.exp(-LOG_NEW))
    AUX_NEW = P_new[:, C:C + A]
    lg_new = np.log(np.clip(P_new[:, :C], 1e-6, 1 - 1e-6) / (1 - np.clip(P_new[:, :C], 1e-6, 1 - 1e-6)))
    Pcal_new = P_new[:, :C].copy()
    for c in CAL_OK:
        Pcal_new[:, c] = CAL_MODEL[c].predict_proba(np.c_[lg_new[:, c], np.zeros(N_NEW), np.zeros(N_NEW)])[:, 1]
    head_n = P_new[:, -2:].clip(1e-4, 1 - 1e-4)
    lgc_n = np.log(np.clip(Pcal_new, 1e-4, 1 - 1e-4) / (1 - np.clip(Pcal_new, 1e-4, 1 - 1e-4)))
    STK_N = np.c_[np.log(head_n / (1 - head_n)), lgc_n[:, :nC], np.full((N_NEW, 4), np.nan), np.full(N_NEW, np.nan)]
    est_n = np.stack([FRAC_MODEL[t][0].predict(np.where(FRAC_MODEL[t][1], 0.0, STK_N)).clip(0, 1) for t in range(2)], 1)
    est_n = est_n / np.maximum(1, est_n.sum(1, keepdims=True))
    PERC_N = 100 * (1 - est_n.sum(1, keepdims=True)) * Pcal_new[:, rest_cls] / (Pcal_new[:, rest_cls].sum(1, keepdims=True) + 1e-9)
    PERC_N = np.insert(PERC_N, 1, 100 * est_n[:, 1], axis=1)
    PERC_N[:, 0] += 100 * est_n[:, 0]
    loc_n = np.c_[(Pcal_new[:, :nC] >= 0.5) | (Pcal_new[:, :nC] == Pcal_new[:, :nC].max(1, keepdims=True)), Pcal_new[:, nC:] >= 0.5]

    # 7. embedding medi normalizzati come in unit.npy (per le proteine più simili)
    UNIT_NEW = np.hstack([u_ / (np.linalg.norm(u_, axis=1, keepdims=True) + 1e-9) for u_ in [BN['esm_mean']] + [BN['t5_mean']] * cfg['USE_T5']])
    UNIT_NEW = (UNIT_NEW / (np.linalg.norm(UNIT_NEW, axis=1, keepdims=True) + 1e-9)).astype(np.float32)

    table_new = pd.DataFrame({'pid': N_PROT + np.arange(N_NEW), 'uniprot_id': new['acc'].to_numpy(), 'gene': gene_n.reindex(new['nome']).to_numpy(),
                              'protein_name': src_n['protein_name'].reindex(new['nome']).fillna('proteina aggiunta').str.split(' \\(').str[0].to_numpy(),
                              'source': 'nuova'})
    table_new[[f'% probabilità {c}' for c in ALL]] = (100 * Pcal_new).round(1)
    table_new['localizzazioni (P ≥ 50%)'] = ['; '.join(np.array(ALL)[m]) for m in loc_n]
    table_new['predette dalla sola sequenza'] = ['; '.join(np.array(CLASSES)[np.maximum(p >= thr_b, p == p.max())]) for p in P_new[:, :nC]]
    table_new['insieme ad alta sensibilità'] = ['; '.join(np.array(CLASSES)[p >= conf_b]) for p in P_new[:, :nC]]
    table_new[[f'ripartizione % {c}' for c in CLASSES]] = PERC_N.round(1)
    table_new['origine frazioni'] = 'predetta'
    table_new['card_protein'] = card_n.reindex(new['nome']).to_numpy()
    table_new = table_new.reindex(columns=table.columns)
    table_new[['misure (fonte: classi)', 'fonti che la misurano', 'partner con etichetta']] = ['', 0, 0]

    # il ragionamento, scritto: dove, perché (meccanismi), a chi somiglia
    U_known = np.load(DATA / 'unit.npy')[table['pid'].to_numpy()].astype(np.float32)
    S_n = UNIT_NEW @ U_known.T
    for i, nome in enumerate(new['nome']):
        vicini = np.argsort(-S_n[i])[:5]
        mec = sorted([(P_new[i, C + AUX.index(a)], l_) for a, l_ in compress(MECC_LAB.items(), [a in AUX for a in MECC_LAB])], reverse=True)[:4]
        print(f"\n■ {nome} ({len(new['sequence'][i])} aa{(', ' + new['acc'][i]) * int(new['acc'][i] != '')})")
        print('  Dove:', ', '.join(f'{ALL[c]} {100 * Pcal_new[i, c]:.0f}%' for c in np.argsort(-Pcal_new[i])[:5]))
        print('  Perché (meccanismi predetti):', ', '.join(f'{l_} {100 * p_:.0f}%' for p_, l_ in mec))
        print('  Ripartizione:', ', '.join(f'{CLASSES[c]} {PERC_N[i, c]:.0f}%' for c in np.argsort(-PERC_N[i])[:3]))
        print('  Più simili nel progetto:', '; '.join(f"{table['gene'].iloc[j]} ({100 * S_n[i, j]:.0f}%: {table['localizzazioni (P ≥ 50%)'].iloc[j]})" for j in vicini))
table_all = pd.concat([table, table_new], ignore_index=True)
table_new.to_excel(DATA / 'proteine_nuove.xlsx', index=False)

## 7 · Varianti: priorità per patogenicità, e un risultato negativo sulla mislocalizzazione
WT e mutante sono predetti dal modello del fold in cui la proteina WT era nel test: nessuna memorizzazione.

**Cosa dicono i dati (da dire così anche nell'articolo).** Sulle varianti con localizzazione misurata al microscopio
(Lacoste et al. 2024) i segnali di *spostamento* della localizzazione (JSD tra i profili, variazione dei meccanismi,
rischi per meccanismo) **non distinguono** le varianti mislocalizzate dalle altre: le AUROC stanno intorno a 0,5 e i
loro intervalli di confidenza contengono 0,5. L'unico segnale è l'LLR di ESM-2, che misura quanto la sostituzione è
sfavorita dall'evoluzione, cioè il danno generico, non lo spostamento. Su ClinVar il punteggio supervisionato funziona,
ma quasi tutto il merito è dell'LLR. Quindi:
- `score_variante` è un punteggio di **priorità per patogenicità**, non una predizione di mislocalizzazione;
- `score_misloc` e i segnali di meccanismo sono **ipotesi da verificare**, non evidenze: servono a dire *quale*
  meccanismo guardare in laboratorio per una variante già sospetta, non *se* la variante mislocalizza;
- predire la mislocalizzazione di una variante dalla sola sequenza resta un problema aperto, con questi dati e questo
  modello.

Segnali per variante:
- **Δ localizzazione**: JSD tra i profili normalizzati (solo come ordinamento), massima variazione negli organelli e
  nei compartimenti neuronali, variazione delle teste di frazione;
- **Δ meccanismi**: variazione delle teste ausiliarie di sorting (peptide segnale, transit peptide, TM, ancora
  lipidica, NLS, palmitoilazione, glicosilazione, superclassi); il "meccanismo principale" è la più spostata tra queste;
- **spostamento locale** (1 − coseno tra embedding ±15 residui di WT e mutante) e **LLR di ESM-2** (log P mutante −
  log P WT nel contesto WT: proxy di destabilizzazione e di vincolo evolutivo);
- **pLDDT del sito** (AlphaFold): una sostituzione in un tratto ordinato e sepolto destabilizza il ripiegamento, in un
  tratto disordinato molto meno;
- **ΔΔG di inserzione in membrana** (scala di Hessa 2005) se il sito cade in una finestra compatibile con un'elica TM;
- **rischi per meccanismo** (domanda 49): ritenzione nel RE = P(via secretoria) × destabilizzazione × ordine del sito;
  nucleo-citoplasma = P(nucleo) × perdita di NLS o Δ nucleo; sinaptico = P(sinapsi) × perdita del ligando PDZ o Δ sinapsi.

Punteggi: **non supervisionato** (`score_misloc`, media dei percentili dei segnali) e **supervisionato**
(`score_variante`: gradient boosting su ClinVar P/LP contro B/LB, CV raggruppata per proteina, solo ciò che cambia con
la mutazione). `score_supervisionato` aggiunge il contesto del gene (espressione, numero di localizzazioni): premia il
gene più che la variante, quindi è riportato solo per confronto e **non** è il punteggio da citare.

Valutazione: AUROC con intervallo di confidenza al 95% (bootstrap sulle varianti), anche **entro la stessa proteina**
(su ClinVar e sulle varianti misurate): è il confronto che toglie l'effetto del gene. Le varianti di ClinVar sono un
campione casuale di 4.000 tra quelle del perimetro (vedi lo 01).


In [ ]:
var_P = 1 / (1 + np.exp(-z['var']))                                          # [fold, riga, uscita]
fpos = pd.Series(np.arange(len(z['folds'])), index=z['folds'])
vi = pd.Series(np.arange(len(var_rows)), index=var_rows)
fold_i = fpos[part[cv['pid'].to_numpy()]].to_numpy()
p_wt, p_mt = var_P[fold_i, vi[cv['pid']].to_numpy()], var_P[fold_i, vi[cv['row']].to_numpy()]
q_wt, q_mt = p_wt[:, :nC] / p_wt[:, :nC].sum(1, keepdims=True), p_mt[:, :nC] / p_mt[:, :nC].sum(1, keepdims=True)
q_m = (q_wt + q_mt) / 2
cv['JSD'] = (0.5 * (q_wt * np.log2(np.maximum(q_wt, 1e-9) / np.maximum(q_m, 1e-9))).sum(1)
             + 0.5 * (q_mt * np.log2(np.maximum(q_mt, 1e-9) / np.maximum(q_m, 1e-9))).sum(1))
cv['max_dP_organelli'] = np.abs(p_mt[:, :nC] - p_wt[:, :nC]).max(1)
cv['max_dP_neuro'] = np.abs(p_mt[:, nC:C] - p_wt[:, nC:C]).max(1)
cv[[f'd_{a}' for a in AUX]] = p_mt[:, C:C + A] - p_wt[:, C:C + A]
SORTING = ['aux_peptide_segnale', 'aux_transit_peptide', 'aux_tm_1piu', 'aux_tm_2piu', 'aux_ancora_lipidica', 'aux_glicoproteina',
           'aux_nls', 'aux_palmitoilazione', 'aux_via_secretoria', 'aux_nucleo_citoplasma', 'aux_import_organelli']
MECC = list(np.array(AUX)[np.isin(AUX, SORTING)])
d_mecc = np.abs(cv[[f'd_{a}' for a in MECC]].to_numpy())
cv['max_d_meccanismo'] = d_mecc.max(1)
cv['meccanismo_principale'] = np.array(MECC)[d_mecc.argmax(1)]
cv['d_frazione_citosolica'] = p_mt[:, -2] - p_wt[:, -2]
cv['d_frazione_nucleare'] = p_mt[:, -1] - p_wt[:, -1]

# vista locale e LLR: un file per-residuo per proteina WT, aperto e chiuso subito (con migliaia di file aperti insieme
# il sistema operativo esaurisce i descrittori)
RES, LOC = Path(cfg['RES_DIR']), np.load(cfg['LOC_FILE'], mmap_mode='r')
aa_i = {a: i for i, a in enumerate('ACDEFGHIKLMNPQRSTVWY')}
wt_i, mt_i, pos_t = cv['wt'].map(aa_i).to_numpy(), cv['mt'].map(aa_i).to_numpy(), cv['pos_t'].to_numpy()
loc_wt, llr = np.full((len(cv), LOC.shape[1]), np.nan, np.float32), np.full(len(cv), np.nan)
gruppi = cv.groupby('pid').indices
for pid, idx in gruppi.items():
    for f_ in compress([RES / f'{pid}.npz'], [(RES / f'{pid}.npz').exists()]):
        with np.load(f_) as npz:
            h, lp = npz['h'].astype(np.float32), npz['lp']
        loc_wt[idx] = np.stack([h[max(q - 1 - WIN, 0):q + WIN].mean(0) for q in pos_t[idx]])
        llr[idx] = lp[pos_t[idx] - 1, mt_i[idx]] - lp[pos_t[idx] - 1, wt_i[idx]]
loc_mt = np.asarray(LOC[cv['row'].to_numpy()], np.float32)
with np.errstate(all='ignore'):
    cv['shift_locale'] = 1 - (loc_wt * loc_mt).sum(1) / (np.linalg.norm(loc_wt, axis=1) * np.linalg.norm(loc_mt, axis=1))
cv['LLR'] = llr
print(f'Varianti con file per-residuo ESM-2: {np.isfinite(llr).mean():.0%}')

# pLDDT del sito e della finestra ±5 dalla struttura AlphaFold della WT (numerazione UniProt completa)
AFD = Path(cfg['AF_DIR'])
# per ogni Cα: numero del residuo, coordinate x, y, z e pLDDT (colonna del fattore B)
CA_RE = re.compile(r'^ATOM\s+\d+\s+C\s+CA\s+\S+\s+\S+\s+\S+\s+\S+\s+(\d+)\s+\S+\s+(\S+)\s+(\S+)\s+(\S+)\s+\S+\s+(\S+)', re.M)
acc_of = proteins['uniprot_id'].fillna('').to_numpy()
pos_full = cv['pos'].to_numpy()
cv['plddt_sito'], cv['plddt_finestra'], cv['contatti_sito'] = np.nan, np.nan, np.nan
for pid, idx in gruppi.items():
    f_ = AFD / f'{acc_of[pid]}.cif'
    for testo in compress([f_], [f_.exists() and f_.stat().st_size > 0]):
        m_ = np.array(CA_RE.findall(testo.read_text(encoding='utf-8', errors='ignore')), float).reshape(-1, 5)
        b_ = np.full(int(m_[:, 0].max(initial=0)) + 2, np.nan)
        b_[m_[:, 0].astype(int)] = m_[:, 4]
        xyz = np.full((len(b_), 3), np.nan)
        xyz[m_[:, 0].astype(int)] = m_[:, 1:4]
        q_ = np.minimum(pos_full[idx], len(b_) - 1)
        # sepoltura del sito: Cα entro 10 Å, esclusi i vicini di sequenza (±2). Un residuo sepolto regge il ripiegamento:
        # è lì che una sostituzione destabilizza la proteina, che il controllo di qualità del RE poi trattiene o degrada
        d_ = np.linalg.norm(xyz[q_][:, None, :] - xyz[None, :, :], axis=2)
        vicini = ((d_ < 10) & (np.abs(np.arange(len(b_))[None, :] - q_[:, None]) > 2)).sum(1)
        cv.iloc[idx, cv.columns.get_loc('contatti_sito')] = np.where(np.isfinite(xyz[q_, 0]), vicini, np.nan)
        cv.iloc[idx, cv.columns.get_loc('plddt_sito')] = b_[q_]
        cv.iloc[idx, cv.columns.get_loc('plddt_finestra')] = [np.nanmean(np.r_[b_[max(q - 5, 1):q + 6], b_[q]]) for q in q_]
print(f"Varianti con pLDDT del sito: {cv['plddt_sito'].notna().mean():.0%} | con la sepoltura del sito: {cv['contatti_sito'].notna().mean():.0%}")

# biofisica di membrana: ΔG apparente di inserzione (Hessa et al., Nature 2005) su una finestra di 19 residui
wt_seq = cv['pid'].map(proteins.set_index('pid')['sequence'])
HESSA = dict(zip('ILFVMCAWTYGSNHPQREKD', [-0.60, -0.55, -0.32, -0.31, -0.10, -0.13, 0.11, 0.30, 0.52, 0.68, 0.74, 0.84,
                                          2.05, 2.06, 2.23, 2.36, 2.58, 2.68, 2.71, 3.49]))
win19 = pd.Series([s[max(p - 10, 0):p + 9] for s, p in zip(wt_seq, cv['pos'])], index=cv.index)
dG_wt = sum(win19.str.count(a) * HESSA[a] for a in HESSA)
in_tm = (dG_wt < 2.0) & (win19.str.len() == 19)
cv['ddG_inserzione_TM'] = (cv['mt'].map(HESSA) - cv['wt'].map(HESSA)) * in_tm
cv['perde_inserzione'] = (in_tm & (dG_wt < 0) & (dG_wt + cv['ddG_inserzione_TM'] > 0)).astype(int)
cv['in_Nterm30'] = (cv['pos'] <= 30).astype(int)
cv['in_Cterm10'] = (cv['pos'] > wt_seq.str.len() - 10).astype(int)
hand_d = hand.iloc[cv['row']].to_numpy() - hand.iloc[cv['pid']].to_numpy()
HC = list(hand.columns)
MOTIFS = ['kdel', 'kkxx', 'pts1', 'caax', 'myr', 'nls', 'pts1_ext', 'pts2', 'yxxphi_ct', 'dileu_ct', 'diacid_ct', 'ff_ct',
          'pdz_I', 'pdz_II', 'nt_cys_palm', 'ankG_bind']
cv['motivi_alterati'] = np.abs(hand_d[:, [HC.index(c) for c in MOTIFS]]).sum(1)
cv['d_nt_charge'] = hand_d[:, HC.index('nt_charge')]
cv['d_nt_muH'] = hand_d[:, HC.index('nt_muH')]                          # rottura dell'elica anfipatica mitocondriale
cv['d_tm_maxlen'] = hand_d[:, HC.index('tm_maxlen')]                    # accorciamento o perdita di un'elica TM
cv['perde_pdz'] = ((hand.iloc[cv['pid']][['pdz_I', 'pdz_II']].to_numpy().max(1) == 1) &
                   (hand.iloc[cv['row']][['pdz_I', 'pdz_II']].to_numpy().max(1) == 0)).astype(int)
cv['perde_nls'] = (hand_d[:, HC.index('nls')] < 0).astype(int)
cv['perde_ankG'] = (hand_d[:, HC.index('ankG_bind')] < 0).astype(int)

# rischi per meccanismo (domanda 49): prior della WT × rottura del segnale corrispondente
destab = pd.Series(-cv['LLR']).rank(pct=True).fillna(0.5).to_numpy()
ordine = (cv['plddt_sito'].fillna(cv['plddt_sito'].median()).fillna(70) / 100).to_numpy()
P_sinapsi = p_wt[:, [ALL.index('Presinapsi'), ALL.index('Postsinapsi')]].max(1)
cv['rischio_ritenzione_RE'] = p_wt[:, C + AUX.index('aux_via_secretoria')] * destab * ordine
# ipotesi di Lacoste et al. 2024: la maggior parte delle mislocalizzazioni patogeniche nasce da un ripiegamento difettoso.
# Segnale definito a priori (non regolato sui dati): sostituzione sfavorita dall'evoluzione in un punto sepolto
sepolto = cv['contatti_sito'].rank(pct=True).fillna(0.5).to_numpy()
cv['rischio_ripiegamento'] = destab * sepolto
cv['rischio_nucleo_citoplasma'] = p_wt[:, CLASSES.index('Nucleus')] * np.maximum(cv['perde_nls'], np.abs(p_mt[:, 1] - p_wt[:, 1]) * 5).clip(0, 1)
cv['rischio_sinaptico'] = P_sinapsi * np.maximum(cv['perde_pdz'], np.abs(p_mt[:, nC + 2:nC + 4] - p_wt[:, nC + 2:nC + 4]).max(1) * 5).clip(0, 1)
cv['rischio_assone'] = p_wt[:, ALL.index('Assone')] * np.maximum(cv['perde_ankG'], np.abs(p_mt[:, nC] - p_wt[:, nC]) * 5).clip(0, 1)
cv = cv.join(proteins.set_index('pid')[CTX], on='pid')

# punteggio non supervisionato: media dei percentili dei segnali di meccanismo (più alto = più sospetto)
MECH = ['JSD', 'max_dP_neuro', 'max_d_meccanismo', 'shift_locale', 'ddG_inserzione_TM', 'motivi_alterati',
        'rischio_ritenzione_RE', 'rischio_nucleo_citoplasma', 'rischio_sinaptico', 'rischio_assone', 'rischio_ripiegamento']
cv['score_misloc'] = np.nanmean(np.c_[cv[MECH].abs().rank(pct=True).to_numpy(), cv['LLR'].rank(pct=True, ascending=False).to_numpy()], 1)

# punteggio supervisionato su ClinVar, CV raggruppata per proteina; la quota di patogeniche del gene NON entra
FEAT = MECH + ['max_dP_organelli', 'LLR', 'plddt_sito', 'plddt_finestra', 'contatti_sito', 'perde_inserzione', 'in_Nterm30', 'in_Cterm10',
               'd_nt_charge', 'd_nt_muH', 'd_tm_maxlen', 'perde_pdz', 'perde_nls', 'perde_ankG', 'd_frazione_citosolica',
               'd_frazione_nucleare'] + [f'd_{a}' for a in AUX] + CTX
pb = cv[cv['clin'].isin(['P/LP', 'B/LB'])]
FEAT = list(pb[FEAT].columns[pb[FEAT].nunique() > 1])
FEAT = list(pd.Index(FEAT)[~pd.Index(FEAT).str.contains('patogeniche')])          # la quota di patogeniche del gene non entra mai
y_pb = (pb['clin'] == 'P/LP').astype(int).to_numpy()
# due versioni. Quella completa usa anche il contesto del gene (espressione, numero di localizzazioni, ruolo): ClinVar è
# sbilanciato per gene (alcuni geni hanno quasi solo patogeniche, altri quasi solo benigne), e questo contesto premia il
# gene più che la variante. Quella "della variante" usa solo ciò che cambia con la mutazione
FEAT_V = list(pd.Index(FEAT).difference(pd.Index(CTX), sort=False))
hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)
for nome_s, F_ in [('score_supervisionato', FEAT), ('score_variante', FEAT_V)]:
    # varianti fuori da P/LP-B/LB (VUS e varianti misurate non classificate): modello stimato su tutte le P/LP e B/LB
    cv[nome_s] = hgb.fit(pb[F_].astype(float), y_pb).predict_proba(cv[F_].astype(float))[:, 1]
    # P/LP e B/LB: fuori fold, con le proteine raggruppate (nessuna proteina sta insieme in training e in valutazione)
    cv.loc[pb.index, nome_s] = cross_val_predict(hgb, pb[F_].astype(float), y_pb, groups=pb['pid'], cv=GroupKFold(5),
                                                 method='predict_proba')[:, 1]

# AUROC con la formula dei ranghi (NaN senza errori se mancano dati); anche per sottogruppi biologici
lv = cv[['gene', 'variante']].reset_index().merge(lab_var.drop_duplicates(['gene', 'variante']), on=['gene', 'variante'])
SCORES = {'score_variante (senza il gene)': cv['score_variante'], 'score_supervisionato (con il gene, solo confronto)': cv['score_supervisionato'],
          'score_misloc': cv['score_misloc'], 'LLR (segno −)': -cv['LLR'], 'rischio_ripiegamento': cv['rischio_ripiegamento'],
          'contatti_sito (sepoltura)': cv['contatti_sito'],
          'JSD': cv['JSD'], 'shift_locale': cv['shift_locale'], 'max_d_meccanismo': cv['max_d_meccanismo'],
          'rischio_ritenzione_RE': cv['rischio_ritenzione_RE'], 'rischio_nucleo_citoplasma': cv['rischio_nucleo_citoplasma'],
          'rischio_sinaptico': cv['rischio_sinaptico'], '|ddG_inserzione_TM|': cv['ddG_inserzione_TM'].abs()}
sec = p_wt[:, C + AUX.index('aux_via_secretoria')] > 0.5
EVAL = {f'ClinVar P/LP vs B/LB (n={len(pb)})': (pb.index.to_numpy(), y_pb),
        'ClinVar, proteine secretorie e di membrana': (pb.index.to_numpy()[sec[pb.index]], y_pb[sec[pb.index]]),
        'ClinVar, proteine nucleari': (pb.index.to_numpy()[p_wt[pb.index, 1] > 0.5], y_pb[p_wt[pb.index, 1] > 0.5]),
        'ClinVar, fenotipo cerebrale': (pb.index.to_numpy()[pb['brain_phenotype'].to_numpy() == 1], y_pb[pb['brain_phenotype'].to_numpy() == 1]),
        f'Mislocalizzazione misurata (n={len(lv)})': (lv['index'].to_numpy(), lv['mislocalizzata'].astype(int).to_numpy())}
auc_v = pd.DataFrame(index=list(SCORES))
for name, (ix, y) in EVAL.items():
    with np.errstate(all='ignore'):
        auc_v[name] = [(pd.Series(v.to_numpy()[ix]).rank().to_numpy()[y == 1].sum() - (y == 1).sum() * ((y == 1).sum() + 1) / 2)
                       / ((y == 1).sum() * (y == 0).sum()) for v in SCORES.values()]
auc_v.loc['varianti (P/LP, B/LB)'] = [f'{(y == 1).sum()}/{(y == 0).sum()}' for ix, y in EVAL.values()]
print('AUROC per sottogruppi (NaN se al segnale manca il valore di qualche variante):')
print(auc_v.to_string())

# le due valutazioni principali con l'intervallo di confidenza al 95%: bootstrap sulle varianti (1.000 ricampionamenti),
# sulle sole varianti in cui il segnale esiste. Un intervallo che contiene 0,5 = il segnale non distingue
from scipy.stats import rankdata
B_V = 1000
PRINCIPALI = [list(EVAL)[0], list(EVAL)[-1]]
ic_rows = []
for name in PRINCIPALI:
    ix, y = EVAL[name]
    for nome_s, v in SCORES.items():
        s_ = v.to_numpy()[ix].astype(float)
        s_ok, y_ok = s_[np.isfinite(s_)], y[np.isfinite(s_)]
        bb = rng.integers(0, max(len(s_ok), 1), (B_V, len(s_ok)))
        with np.errstate(all='ignore'):
            n1_b = (y_ok[bb] == 1).sum(1)
            auc_b = ((rankdata(s_ok[bb], axis=1) * (y_ok[bb] == 1)).sum(1) - n1_b * (n1_b + 1) / 2) / (n1_b * (len(s_ok) - n1_b))
            auc_p = ((rankdata(s_ok) * (y_ok == 1)).sum() - (y_ok == 1).sum() * ((y_ok == 1).sum() + 1) / 2) / ((y_ok == 1).sum() * (y_ok == 0).sum())
            ic_rows.append({'valutazione': name, 'segnale': nome_s, 'varianti': len(s_ok), 'AUROC': auc_p,
                            'IC 2,5%': np.nanquantile(auc_b, 0.025), 'IC 97,5%': np.nanquantile(auc_b, 0.975)})
auc_ic = pd.DataFrame(ic_rows)
auc_ic['distingue (IC esclude 0,5)'] = np.where((auc_ic['IC 2,5%'] > 0.5) | (auc_ic['IC 97,5%'] < 0.5), 'sì', 'no')
print('\nAUROC con IC 95% (bootstrap sulle varianti):')
print(auc_ic.round(3).to_string(index=False))

# confronto entro la stessa proteina: mislocalizza (ed è patogenica) più spesso una proteina secretoria che una nucleare,
# e ClinVar è sbilanciato per gene, per cui l'AUROC sull'insieme mescola la proteina con la variante. Qui conta solo
# l'ordine tra varianti della stessa proteina (media delle AUROC per proteina, sulle proteine con varianti dei due tipi)
auc_entro = {}
for name in PRINCIPALI:
    ix_m, y_m = EVAL[name]
    dm = pd.DataFrame({'g': cv['pid'].to_numpy()[ix_m], 'y': y_m})
    for nome, v in SCORES.items():
        d_ = dm.assign(s=v.to_numpy()[ix_m]).dropna()
        d_['r'] = d_.groupby('g')['s'].rank()
        st = d_.assign(rp=d_['r'] * d_['y']).groupby('g').agg(n1=('y', 'sum'), n=('y', 'size'), rp=('rp', 'sum'))
        st = st[(st['n1'] > 0) & (st['n1'] < st['n'])]
        auc_entro[(name, nome)] = [((st['rp'] - st['n1'] * (st['n1'] + 1) / 2) / (st['n1'] * (st['n'] - st['n1']))).mean(), len(st)]
entro = pd.DataFrame(auc_entro, index=['AUROC entro proteina', 'proteine']).T
print('\nAUROC entro la stessa proteina (media sulle proteine con varianti di entrambi i tipi):')
print(entro.round(3).to_string())

# il verdetto, scritto dai numeri: quali segnali distinguono le varianti mislocalizzate
mis = auc_ic[auc_ic['valutazione'] == PRINCIPALI[-1]].set_index('segnale')
SPOSTAMENTO = ['score_misloc', 'JSD', 'shift_locale', 'max_d_meccanismo', 'rischio_ritenzione_RE', 'rischio_nucleo_citoplasma', 'rischio_sinaptico',
               '|ddG_inserzione_TM|']
print(f"\nMislocalizzazione misurata ({int(mis['varianti'].max())} varianti): segnali di spostamento il cui IC esclude 0,5 verso l'alto: "
      f"{list(mis.index[mis.index.isin(SPOSTAMENTO) & (mis['IC 2,5%'] > 0.5)])} su {len(SPOSTAMENTO)} | "
      f"LLR di ESM-2: {mis.loc['LLR (segno −)', 'AUROC']:.3f} (IC {mis.loc['LLR (segno −)', 'IC 2,5%']:.3f}–{mis.loc['LLR (segno −)', 'IC 97,5%']:.3f})")
print('Lettura: se la lista è vuota (o quasi) la predizione dello spostamento è un risultato negativo; i punteggi restano una priorità per patogenicità.')
with pd.ExcelWriter(DATA / 'varianti_valutazione.xlsx') as xw:
    auc_ic.round(4).to_excel(xw, sheet_name='AUROC con IC', index=False)
    entro.round(4).to_excel(xw, sheet_name='entro proteina')
    auc_v.to_excel(xw, sheet_name='sottogruppi')
print('\nMeccanismo principale nelle P/LP più sospette (top 10% di score_misloc):')
print(cv[(cv['clin'] == 'P/LP') & (cv['score_misloc'] >= cv['score_misloc'].quantile(0.9))]['meccanismo_principale'].value_counts().head(8).to_string())
cv.sort_values('score_misloc', ascending=False).to_excel(DATA / 'varianti_mislocalizzazione.xlsx', index=False)

## 8 · Malattie neurologiche: in quali compartimenti cadono le loro proteine?
Le proteine si raggruppano per malattia dal testo delle malattie di UniProt (colonna `malattie_uniprot`; i gruppi si
riconoscono dai nomi clinici, come Leigh o Niemann-Pick, non dalle parole di localizzazione). Le probabilità usate sono
quelle della **sola sequenza, fuori fold** (non quelle fuse con le misure, che per le proteine annotate ripeterebbero
le annotazioni): la domanda è se il modello, senza conoscere la proteina, la mette dove la neurologia si aspetta. Per
ogni gruppo e compartimento: AUC = probabilità che una proteina del gruppo abbia una probabilità più alta di una proteina cerebrale fuori dal gruppo (0,5 = nessuna differenza), test di Mann-Whitney con
approssimazione normale e correzione di Benjamini-Hochberg su tutti i confronti. Gruppi con meno di 15 proteine: NaN.
La tabella sotto la mappa confronta i risultati con le associazioni attese dalla neurologia (domanda 54). Le attese non
ritrovate, o trovate nel verso opposto, vanno riportate come tali: sono un limite del modello (o dell'attesa), non un
dettaglio da omettere. Le classi con meno di 30 positivi non sono valutabili.

In [ ]:
disease = proteins.get('malattie_uniprot', proteins['card_protein'].str.extract(r'Disease: (.*?)(?=Brain expression \(Human Protein Atlas\)|$)',
                                                                             expand=False)).fillna('')
P_dis = Pb[:, :C]                                  # sola sequenza, fuori fold: mai le probabilità fuse con le misure
GROUPS = {'Neurodegenerazione': r'alzheimer|parkinson|amyotrophic|frontotemporal|huntington|lewy|prion|creutzfeldt|neurodegenerat|spinocerebellar|ataxia',
          'Epilessie': r'epilep|seizure|dravet|convuls',
          'Neurosviluppo (DI, autismo)': r'intellectual disability|autism|developmental delay|neurodevelopmental|rett syndrome|angelman|microcephal',
          'Leucodistrofie e mielina': r'leukodystroph|leukoencephalopath|hypomyelinat|pelizaeus|demyelinat|white matter',
          'Neuropatie periferiche': r'charcot-marie-tooth|hereditary motor and sensory|neuropathy',
          'Encefalopatie energetiche': r'leigh|encephalomyopath|melas|merrf|oxidative phosphorylation|complex [iv]+ deficiency',
          'Malattie da accumulo': r'storage disease|ceroid lipofuscinosis|gaucher|niemann|krabbe|sanfilippo|mucopolysacchar|zellweger|refsum|tay-sachs|sandhoff|fabry'}
bm = np.flatnonzero(bmask)
rows_d = []
for gname, pat in GROUPS.items():
    in_g = disease.to_numpy()[bm].astype(str)
    in_g = pd.Series(in_g).str.contains(pat, case=False).to_numpy()
    for c in range(C):
        ok = gate[bm, c]
        x, y = P_dis[bm[ok & in_g], c], P_dis[bm[ok & ~in_g], c]
        r_ = pd.Series(np.r_[x, y]).rank().to_numpy()
        n1, n2 = len(x), len(y)
        with np.errstate(all='ignore'):
            U = r_[:n1].sum() - n1 * (n1 + 1) / 2
            zst = (U - n1 * n2 / 2) / np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
        rows_d.append({'gruppo': gname, 'compartimento': ALL[c], 'proteine': n1,
                       'AUC': np.where(n1 >= 15, U / max(n1 * n2, 1), np.nan), 'p': np.where(n1 >= 15, 2 * norm.sf(np.abs(zst)), np.nan)})
dis = pd.DataFrame(rows_d).astype({'AUC': float, 'p': float})
dis['q (BH)'] = (dis.sort_values('p')['p'] * dis['p'].notna().sum() / np.arange(1, len(dis) + 1)).iloc[::-1].cummin().iloc[::-1].reindex(dis.index).clip(upper=1)
mappa = dis.pivot(index='gruppo', columns='compartimento', values='AUC').reindex(index=list(GROUPS), columns=ALL)
qmap = dis.pivot(index='gruppo', columns='compartimento', values='q (BH)').reindex(index=list(GROUPS), columns=ALL)
n_g = dis.groupby('gruppo')['proteine'].max().reindex(list(GROUPS))

fig, ax = plt.subplots(figsize=(15, 5.2))
im = ax.imshow(mappa.to_numpy() - 0.5, cmap='RdBu_r', vmin=-0.3, vmax=0.3, aspect='auto')
for (r_i, c_i), v in np.ndenumerate(mappa.to_numpy()):
    ax.text(c_i, r_i, f'{v:.2f}{"*" * int(qmap.to_numpy()[r_i, c_i] < 0.05)}', ha='center', va='center', fontsize=8,
            color=['#222222', '#ffffff'][int(abs(np.nan_to_num(v) - 0.5) > 0.2)])
ax.set_xticks(np.arange(C), ALL, fontsize=8, rotation=35, ha='right')
ax.set_yticks(np.arange(len(GROUPS)), [f'{g} (n={n})' for g, n in n_g.items()], fontsize=9)
fig.colorbar(im, ax=ax, label='AUC − 0,5 (rosso = il gruppo sta di più in quel compartimento)')
ax.set_title('Proteine delle malattie neurologiche contro le altre proteine del cervello (* = FDR < 5%)', fontsize=12)
fig.tight_layout()
fig.savefig(DATA / 'malattie_compartimenti.png', dpi=150)
plt.show()

ATTESE = [('Epilessie', 'Cell membrane'), ('Epilessie', 'Presinapsi'), ('Neurosviluppo (DI, autismo)', 'Nucleus'),
          ('Neurosviluppo (DI, autismo)', 'Postsinapsi'), ('Leucodistrofie e mielina', 'Mielina'), ('Leucodistrofie e mielina', 'Peroxisome'),
          ('Leucodistrofie e mielina', 'Lysosome/Vacuole'), ('Encefalopatie energetiche', 'Mitochondrion'),
          ('Malattie da accumulo', 'Lysosome/Vacuole'), ('Neurodegenerazione', 'Cytoplasm'), ('Neuropatie periferiche', 'Assone')]
attese = pd.DataFrame([{'gruppo': g, 'compartimento atteso': c, 'AUC': mappa.loc[g, c], 'q (BH)': qmap.loc[g, c]} for g, c in ATTESE])
# una classe con meno di 30 positivi di riferimento (la mielina ne ha 7) non ha un predittore affidabile: l'attesa non è verificabile
pos_cls = pd.Series(n_ref[:, 1], index=ALL)
attese['positivi della classe'] = attese['compartimento atteso'].map(pos_cls)
attese['ritrovata'] = np.select([attese['AUC'].isna().to_numpy() | (attese['positivi della classe'] < MIN_POS).to_numpy(),
                                 ((attese['AUC'] > 0.5) & (attese['q (BH)'] < 0.05)).to_numpy(),
                                 ((attese['AUC'] < 0.5) & (attese['q (BH)'] < 0.05)).to_numpy()],
                                ['non valutabile', 'sì', 'no, verso opposto'], 'no')
print(attese.round(3).to_string(index=False))
print(f"Attese ritrovate: {(attese['ritrovata'] == 'sì').sum()} su {(attese['ritrovata'] != 'non valutabile').sum()} valutabili; "
      f"nel verso opposto: {(attese['ritrovata'] == 'no, verso opposto').sum()}")
dis.round(4).to_excel(DATA / 'malattie_compartimenti.xlsx', index=False)

## 9 · Casi di studio
Proteine al centro delle malattie neurologiche più studiate, con le % di probabilità, le misure note e le varianti
ClinVar più sospette. Servono a leggere il modello con gli occhi del clinico: TDP-43 e FUS devono avere un nucleo alto
con una quota citoplasmatica, PSD-95 e SHANK3 la postsinapsi, PLP1 e MBP la mielina, PINK1 il mitocondrio, ABCD1 e PEX7
il perossisoma, NPC1 e GBA1 il lisosoma, SCN1A e KCNQ2 la membrana (e l'assone).

In [ ]:
GENI = ['TARDBP', 'FUS', 'SOD1', 'MAPT', 'SNCA', 'APP', 'PSEN1', 'LRRK2', 'PINK1', 'PRKN', 'GBA1', 'NPC1', 'HTT', 'SHANK3', 'SYNGAP1',
        'DLG4', 'GRIN2B', 'SCN1A', 'KCNQ2', 'MECP2', 'PLP1', 'MBP', 'MOG', 'ABCD1', 'PEX7', 'GRN', 'TMEM106B', 'ATXN2']
pato = cv[cv['clin'] == 'P/LP']
var_g = pd.DataFrame({'varianti P/LP': pato.groupby('gene').size(),
                      'più sospetta': pato.sort_values('score_variante', ascending=False).drop_duplicates('gene').set_index('gene')['variante'],
                      'meccanismo': pato.sort_values('score_variante', ascending=False).drop_duplicates('gene').set_index('gene')['meccanismo_principale']})
COLS = [f'% probabilità {c}' for c in ['Cytoplasm', 'Nucleus', 'Cell membrane', 'Mitochondrion', 'Endoplasmic reticulum',
                                     'Lysosome/Vacuole', 'Peroxisome', 'Assone', 'Dendrite', 'Presinapsi', 'Postsinapsi', 'Mielina']]
casi = table.drop_duplicates('gene').set_index('gene').reindex(GENI)[COLS + ['localizzazioni (P ≥ 50%)', 'misure (fonte: classi)']]
casi = casi.join(var_g)
casi.columns = casi.columns.str.replace('% probabilità ', '')
print(f'Geni non presenti nella tabella del cervello: {list(casi.index[casi.iloc[:, 0].isna()])}')
print(casi.round(0).to_string())
casi.to_excel(DATA / 'casi_di_studio.xlsx')

## 10 · Curve ROC finali: compartimenti neuronali e varianti
A sinistra, per ogni compartimento del neurone con almeno 30 positivi (out-of-fold, proteine delle cellule giuste; la
mielina resta fuori), la ROC della probabilità tarata con i partner contro la rete da sola. Sono positivi annotati
contro proteine non annotate, valutati sulle stesse fonti dell'addestramento: non è una validazione indipendente. A destra i
punteggi delle varianti su ClinVar (P/LP contro B/LB) e, se ci sono, sulle varianti con localizzazione misurata.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5))
COLN = ['#eb6834', '#2a78d6', '#1baf7a', '#7b4bd6', '#c79a1b']
for c, col in zip(range(nC, C), COLN):
    y_c = lab_cal[ref_h[c], c]
    for _ in range(int((np.nansum(y_c) >= MIN_POS) and ((y_c == 0).sum() >= MIN_POS))):
        for P_, ls, suff in [(Pcal, '-', 'rete + partner'), (Pb[:, :C], ':', 'rete')]:
            fpr, tpr, _ = roc_curve(y_c, P_[ref_h[c], c])
            ax1.plot(fpr, tpr, color=col, ls=ls, lw=2, label=f'{ALL[c]}, {suff} (AUC {roc_auc_score(y_c, P_[ref_h[c], c]):.3f}, n+ {int(y_c.sum())})')
ax1.plot([0, 1], [0, 1], color='#c3c2b7', lw=1)
ax1.set(xlabel='Falsi positivi (FPR)', ylabel='Veri positivi (TPR)', title='Compartimenti del neurone (out-of-fold, annotate contro non annotate)')
ix, y = EVAL[list(EVAL)[0]]
for (nome, v), col in zip(list(SCORES.items())[:6], ['#eb6834', '#2a78d6', '#d62a2a', '#1baf7a', '#7b4bd6', '#c79a1b']):
    ok_v = np.isfinite(v.to_numpy()[ix])
    for _ in range(int(ok_v.sum() > 10 and len(set(y[ok_v])) == 2)):
        fpr, tpr, _ = roc_curve(y[ok_v], v.to_numpy()[ix][ok_v])
        ax2.plot(fpr, tpr, color=col, lw=2, label=f'{nome} (AUC {roc_auc_score(y[ok_v], v.to_numpy()[ix][ok_v]):.3f})')
ix_l, y_l = EVAL[list(EVAL)[-1]]
for nome, ls in [('rischio_ripiegamento', '--'), ('LLR (segno −)', ':')]:
    v_l = SCORES[nome].to_numpy()[ix_l]
    ok_l = np.isfinite(v_l)
    for _ in range(int(ok_l.sum() > 10 and len(set(y_l[ok_l])) == 2)):
        fpr, tpr, _ = roc_curve(y_l[ok_l], v_l[ok_l])
        ax2.plot(fpr, tpr, color='#111111', ls=ls, lw=2,
                 label=f'{nome} su mislocalizzazione misurata (AUC {roc_auc_score(y_l[ok_l], v_l[ok_l]):.3f}, n={int(ok_l.sum())})')
ax2.plot([0, 1], [0, 1], color='#c3c2b7', lw=1)
ax2.set(xlabel='Falsi positivi (FPR)', title=f'Varianti: {list(EVAL)[0]}')
for ax in (ax1, ax2):
    ax.legend(fontsize=7.5, loc='lower right', frameon=False)
    ax.grid(color='#e6e5e0', lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('NeuroLoc-Δ: curve ROC finali nel cervello', fontsize=13)
fig.tight_layout()
fig.savefig(DATA / 'roc_finale_cervello.png', dpi=150)
plt.show()
print('File scritti: probabilita_proteine_cervello.xlsx/.parquet, varianti_mislocalizzazione.xlsx, malattie_compartimenti.xlsx, '
      'casi_di_studio.xlsx, calibrazione.png, roc_frazioni.png, malattie_compartimenti.png, roc_finale_cervello.png')

## 11 · La pagina HTML: tutte le proteine, cercabili, in locale
Il notebook scrive `data/neuroloc_delta.html`: un solo file, senza server e senza internet (si apre con doppio clic).
Contiene tutte le proteine della tabella del cervello più quelle nuove della sezione 6b. Per ognuna:
- **Predizione**: % di probabilità per i 9 organelli e i 5 compartimenti neuronali, ripartizione % delle copie (con
  citosol e nucleo misurati quando esistono), localizzazioni, insieme ad alta sensibilità, misure già note;
- **Scheda biologica**: la card_protein, l'espressione nel cervello, i partner molecolari e il profilo dei meccanismi
  di smistamento predetti (radar);
- **Proteine simili**: le 8 proteine più vicine per embedding di sequenza, con le loro **card_protein** e localizzazioni
  (è il ragionamento per analogia di un biologo: "a cosa somiglia, e dove stanno quelle?");
- **Patologia e varianti**: malattie (UniProt), gruppi di malattie neurologiche, varianti ClinVar più sospette e il
  meccanismo che rompono, con le conseguenze attese di una mislocalizzazione per i meccanismi più forti della proteina.

In più, la sezione **Aggiungi una proteina** analizza nel browser una sequenza incollata: segnali di smistamento
(peptide segnale, presequenza, eliche TM, NLS, KDEL, PTS1, ligandi PDZ, ankirina-G…), omologhi tra le proteine del
progetto (5-meri condivisi, come nello 02), stima delle % per omologia e card_protein delle più simili. È una stima
rapida: la predizione completa (rete, calibrazione, percentuali) si ottiene mettendo la sequenza in
`data/esterni/nuove/` e rilanciando lo 03, e la proteina compare nella pagina come tutte le altre.

La sezione **Confronto** mette fianco a fianco le proteine scelte (mappa di calore e radar dei meccanismi).

In [ ]:
# 11 · dati per la pagina: una riga per proteina (tabella del cervello + proteine nuove)
R = table_all.reset_index(drop=True)
pid_k = R['pid'].to_numpy().astype(int)
is_new = (R['source'] == 'nuova').to_numpy()
U_all = np.vstack([np.load(DATA / 'unit.npy')[pid_k[~is_new]].astype(np.float32), UNIT_NEW])
AUX_ALL = np.vstack([1 / (1 + np.exp(-LOGIT[pid_k[~is_new], C:C + A])), AUX_NEW])
SEQ_ALL = np.r_[proteins['sequence'].to_numpy()[pid_k[~is_new]], new['sequence'].to_numpy()]

# le 8 proteine più simili per embedding (coseno), a blocchi per non riempire la memoria
SIM_K = 8
sim_i, sim_s = np.zeros((len(R), SIM_K), int), np.zeros((len(R), SIM_K), np.float32)
for c_ in np.array_split(np.arange(len(R)), max(1, len(R) // 2048)):
    S_ = U_all[c_] @ U_all.T
    S_[np.arange(len(c_)), c_] = -np.inf
    top = np.argpartition(-S_, SIM_K, axis=1)[:, :SIM_K]
    sim_i[c_] = np.take_along_axis(top, np.argsort(-np.take_along_axis(S_, top, 1), 1), 1)
    sim_s[c_] = np.take_along_axis(S_, sim_i[c_], 1)

# partner molecolari, varianti, malattie
g_of = proteins.set_index('pid')['gene']
gp = grafo[grafo['pid_a'].isin(pid_k)].drop_duplicates(['pid_a', 'pid_b'])
partner_d = gp.assign(g=gp['pid_b'].map(g_of)).dropna(subset=['g']).groupby('pid_a')['g'].agg(list).str[:12].to_dict()
cvs = cv.sort_values('score_variante', ascending=False)
cvs = cvs.assign(txt=cvs['variante'] + ' · ' + cvs['clin'] + ' · sospetto ' + (100 * cvs['score_variante']).round().astype('Int64').astype(str)
                 + '% · ' + cvs['meccanismo_principale'].map(MECC_LAB).fillna(cvs['meccanismo_principale']))
var_d = cvs.groupby('pid').head(5).groupby('pid')['txt'].agg(list).to_dict()
nplp_d = cv[cv['clin'] == 'P/LP'].groupby('pid').size().to_dict()
# malattie: il testo intero di UniProt (colonna malattie_uniprot dello 01); la scheda ripulita ne ha solo una parte
dz_scheda = R['card_protein'].fillna('').str.extract(r'Disease: (.*?)(?=Brain expression \(Human Protein Atlas\)|Length:|$)', expand=False).fillna('')
dz_intero = pd.Series(np.where(is_new, '', proteins.get('malattie_uniprot', pd.Series('', index=proteins.index)).fillna('').to_numpy()[np.clip(pid_k, 0, N_PROT - 1)]))
dz = dz_intero.where(dz_intero != '', dz_scheda)
dg = pd.DataFrame({g_: dz.str.contains(p_, case=False) for g_, p_ in GROUPS.items()}).dot(pd.Index(list(GROUPS)) + '|').str.rstrip('|')

MEC_KEYS = list(compress(MECC_LAB, [a in AUX for a in MECC_LAB]))
num = R[[f'% probabilità {c}' for c in ALL]].to_numpy(float)
EXPR = R.reindex(columns=['brain_ntpm', 'neuron_ncpm', 'glia_ncpm', 'oligo_ncpm']).to_numpy(float)
FONTE = {'deeploc': 'DeepLoc (oro)', 'human_extra': 'proteina umana aggiunta', 'hpa_test': 'test HPA', 'nuova': 'aggiunta da te'}
H = pd.DataFrame({
    'g': R['gene'].fillna(R['uniprot_id']).fillna('?'), 'n': R['protein_name'].fillna('').str.split(' \\(').str[0].str.slice(0, 140),
    'u': R['uniprot_id'].fillna(''), 's': R['source'].map(FONTE).fillna(R['source']), 'ne': R['is_neuronal'], 'ol': R['is_oligo'],
    'p': np.where(np.isnan(num), None, np.round(num)).tolist(),
    'l': R['localizzazioni (P ≥ 50%)'].fillna(''), 'sp': R['predette dalla sola sequenza'].fillna(''), 'cf': R['insieme ad alta sensibilità'].fillna(''),
    'r': np.round(R[[f'ripartizione % {c}' for c in CLASSES]].to_numpy(float), 1).tolist(), 'of': R['origine frazioni'].fillna(''),
    'ms': R['misure (fonte: classi)'].fillna(''), 'nf': R['fonti che la misurano'].fillna(0).astype(int),
    'ex': np.where(np.isnan(EXPR), None, np.round(EXPR, 1)).tolist(), 'bs': R.reindex(columns=['brain_specificity'])['brain_specificity'].fillna(''),
    'm': np.round(100 * AUX_ALL[:, [AUX.index(a) for a in MEC_KEYS]]).astype(int).tolist(),
    'si': np.dstack([sim_i, np.round(100 * sim_s).astype(int)]).tolist(),
    'pa': [partner_d.get(p_, []) for p_ in pid_k], 'va': [var_d.get(p_, []) for p_ in pid_k], 'np': [nplp_d.get(p_, 0) for p_ in pid_k],
    'dz': dz.str.slice(0, 1500), 'dg': dg, 'c': R['card_protein'].fillna(''), 'q': SEQ_ALL})
NOMI_IT = {'Cytoplasm': 'Citoplasma', 'Nucleus': 'Nucleo', 'Extracellular': 'Extracellulare', 'Cell membrane': 'Membrana plasm.',
           'Mitochondrion': 'Mitocondrio', 'Endoplasmic reticulum': 'Reticolo endopl.', 'Lysosome/Vacuole': 'Lisosoma',
           'Golgi apparatus': 'Golgi', 'Peroxisome': 'Perossisoma'}
scelta = pd.concat([pd.read_excel(f, index_col=0) for f in sorted(DATA.glob('scelta_metodologia.xlsx'))] + [pd.DataFrame(columns=['AUC macro (test)'])])
META = {'cls': [NOMI_IT.get(c, c) for c in ALL], 'cls_en': ALL, 'nC': nC, 'rip': [NOMI_IT[c] for c in CLASSES],
        'mec': [MECC_LAB[a] for a in MEC_KEYS], 'mec_key': MEC_KEYS, 'best': cfg['BEST'],
        'auc_best': float(scelta['AUC macro (test)'].reindex([cfg['BEST']]).fillna(np.nan).iloc[0]),
        'n_test': cfg['n_test'], 'n': int(len(R)), 'n_new': int(is_new.sum()), 'gruppi': list(GROUPS)}
DATI = json.dumps({'cols': list(H.columns), 'rows': json.loads(H.to_json(orient='values', force_ascii=False))}, ensure_ascii=False)
print(f'Proteine nella pagina: {len(R):,} (nuove: {is_new.sum()}) | dati: {len(DATI) / 1e6:.1f} MB')

In [ ]:
# 11 · la pagina: un file HTML autosufficiente (niente librerie esterne: si apre anche senza internet)
PAGINA = r'''<!DOCTYPE html>
<html lang="it">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NeuroLoc-Δ</title>
<style>
:root{--bg:#f5f5f7;--surface:#fff;--surface2:#f0f0f5;--border:#d2d2d7;--text:#1d1d1f;--text2:#6e6e73;--text3:#8e8e93;
--blue:#0071e3;--blue-l:#e8f0fe;--teal:#00a39b;--teal-l:#e0faf9;--red:#d70015;--red-l:#fff0ef;--orange:#c93400;--orange-l:#fff5e6;
--purple:#8944ab;--purple-l:#f5eeff;--green:#248a3d;--green-l:#edfff3;--shadow:0 2px 8px rgba(0,0,0,.08),0 0 1px rgba(0,0,0,.04);
--radius:16px;--radius-sm:10px;--hdr1:#1d1d1f;--hdr2:#2d2d35}
@media (prefers-color-scheme:dark){:root{--bg:#0f0f12;--surface:#1c1c20;--surface2:#26262c;--border:#3a3a40;--text:#f2f2f7;--text2:#a1a1a8;
--text3:#7c7c84;--blue:#4c9dff;--blue-l:#14243d;--teal:#3fd6cc;--teal-l:#10302e;--red:#ff6961;--red-l:#3a1614;--orange:#ffb340;
--orange-l:#3a2810;--purple:#c58af2;--purple-l:#2a1a38;--green:#4cd964;--green-l:#11301a;--hdr1:#000;--hdr2:#16161c}}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--bg);color:var(--text);font-family:-apple-system,'SF Pro Text','Helvetica Neue',Arial,sans-serif;font-size:15px;line-height:1.6;-webkit-font-smoothing:antialiased}
.header{background:linear-gradient(135deg,var(--hdr1),var(--hdr2));color:#fff;padding:48px 24px 40px;text-align:center}
.logo{font-size:.78rem;letter-spacing:.15em;text-transform:uppercase;color:#3fd6cc;font-weight:600;margin-bottom:10px}
.header h1{font-size:clamp(1.8rem,4.5vw,2.6rem);font-weight:700;letter-spacing:-.02em;line-height:1.15;margin-bottom:10px;
background:linear-gradient(135deg,#fff,#3fd6cc);-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text}
.subtitle{font-size:1rem;color:rgba(255,255,255,.66);max-width:720px;margin:0 auto 26px}
.search{position:relative;max-width:640px;margin:0 auto 22px}
.search input{width:100%;padding:15px 20px 15px 46px;border-radius:14px;border:1px solid rgba(255,255,255,.18);background:rgba(255,255,255,.96);
color:#1d1d1f;font:inherit;font-size:1.02rem;outline:none;box-shadow:0 8px 24px rgba(0,0,0,.25)}
.search input:focus{box-shadow:0 0 0 4px rgba(63,214,204,.45)}
.search svg{position:absolute;left:16px;top:50%;transform:translateY(-50%);color:#8e8e93}
.results{position:absolute;left:0;right:0;top:calc(100% + 6px);background:var(--surface);color:var(--text);border-radius:12px;box-shadow:0 12px 32px rgba(0,0,0,.25);
max-height:380px;overflow:auto;text-align:left;z-index:20;display:none}
.results.open{display:block}
.res{padding:10px 16px;cursor:pointer;display:flex;gap:10px;align-items:baseline;border-bottom:1px solid var(--surface2)}
.res:hover,.res.sel{background:var(--blue-l)}
.res b{min-width:90px}.res span{color:var(--text2);font-size:.86rem;flex:1;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.res em{font-style:normal;font-size:.72rem;color:var(--text3)}
.class-legend{display:inline-flex;gap:20px;flex-wrap:wrap;justify-content:center;background:rgba(255,255,255,.06);border:1px solid rgba(255,255,255,.12);
border-radius:14px;padding:12px 20px}
.legend-item{display:flex;align-items:center;gap:8px;font-size:.8rem;color:rgba(255,255,255,.72)}
.legend-dot{width:10px;height:10px;border-radius:50%}
main{max-width:1120px;margin:0 auto;padding:28px 16px 80px}
.modes{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:22px}
.mode{padding:8px 18px;border-radius:20px;border:1.5px solid var(--border);background:var(--surface);color:var(--text2);font:inherit;font-size:.86rem;font-weight:600;cursor:pointer}
.mode:hover{border-color:var(--blue);color:var(--blue)}
.mode.active{background:var(--blue);border-color:var(--blue);color:#fff}
.view{display:none}.view.active{display:block;animation:fadeIn .25s ease}
@keyframes fadeIn{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}
.section-title{font-size:1.35rem;font-weight:700;letter-spacing:-.01em;margin-bottom:4px}
.section-sub{font-size:.88rem;color:var(--text2);margin-bottom:20px}
.chips{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:18px}
.chip{padding:5px 14px;border-radius:20px;border:1.5px solid var(--border);background:var(--surface);color:var(--text2);font:inherit;font-size:.8rem;font-weight:600;cursor:pointer}
.chip:hover{border-color:var(--blue);color:var(--blue)}
.protein-card{background:var(--surface);border-radius:var(--radius);box-shadow:var(--shadow);overflow:hidden}
.card-header{padding:24px 28px 18px;border-bottom:1px solid var(--border);display:flex;justify-content:space-between;gap:16px;flex-wrap:wrap}
.card-gene{font-size:.74rem;font-weight:700;letter-spacing:.1em;text-transform:uppercase;color:var(--blue);margin-bottom:2px}
.card-name{font-size:1.35rem;font-weight:700;letter-spacing:-.01em;margin-bottom:4px}
.badges{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}
.badge{font-size:.72rem;font-weight:600;padding:3px 10px;border-radius:20px;white-space:nowrap;border:1px solid transparent}
.b-blue{background:var(--blue-l);color:var(--blue)}.b-purple{background:var(--purple-l);color:var(--purple)}.b-teal{background:var(--teal-l);color:var(--teal)}
.b-red{background:var(--red-l);color:var(--red)}.b-orange{background:var(--orange-l);color:var(--orange)}.b-grey{background:var(--surface2);color:var(--text2);border-color:var(--border)}
.b-green{background:var(--green-l);color:var(--green)}
.side{text-align:right;flex-shrink:0}
.info-label{font-size:.68rem;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:var(--text3);margin-bottom:2px}
.info-value{font-size:.88rem;font-weight:600}
.btn{padding:7px 14px;border-radius:10px;border:1.5px solid var(--border);background:var(--surface);color:var(--text);font:inherit;font-size:.8rem;font-weight:600;cursor:pointer}
.btn:hover{border-color:var(--blue);color:var(--blue)}
.btn.primary{background:var(--blue);border-color:var(--blue);color:#fff}
.tabs{display:flex;border-bottom:1px solid var(--border);padding:0 20px;overflow-x:auto}
.tab{padding:12px 16px;font:inherit;font-size:.84rem;font-weight:600;color:var(--text2);cursor:pointer;border:none;background:none;border-bottom:2.5px solid transparent;margin-bottom:-1px;white-space:nowrap}
.tab:hover{color:var(--text)}.tab.active{color:var(--blue);border-bottom-color:var(--blue)}
.pane{display:none;padding:24px 28px}.pane.active{display:block}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:24px}
@media(max-width:760px){.grid2{grid-template-columns:1fr}.card-header{padding:20px 16px;flex-direction:column}.pane{padding:18px 16px}.side{text-align:left}}
.stitle{font-size:.7rem;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:var(--text3);margin-bottom:10px}
.bar{display:flex;align-items:center;gap:10px;margin-bottom:7px}
.bar .nm{font-size:.8rem;width:128px;flex-shrink:0}
.track{flex:1;height:8px;background:var(--surface2);border-radius:4px;overflow:hidden}
.fill{height:100%;border-radius:4px;transition:width .6s cubic-bezier(.23,1,.32,1)}
.bar .pc{font-size:.78rem;font-weight:600;width:40px;text-align:right}
.bar.dom .nm{font-weight:700}
.bar.na{opacity:.38}
.box{background:var(--surface2);border-radius:var(--radius-sm);padding:16px 18px;margin-top:20px}
.tags{display:flex;flex-wrap:wrap;gap:6px;margin-top:14px;padding-top:14px;border-top:1px solid var(--border)}
.tags .stitle{width:100%;margin-bottom:2px}
.note{font-size:.78rem;color:var(--text2);margin-top:8px}
.cardtxt{font-size:.86rem;line-height:1.75;color:var(--text)}
.cardtxt strong{color:var(--text)}
.sim{border:1px solid var(--border);border-radius:var(--radius-sm);padding:14px 16px;margin-bottom:12px;background:var(--surface)}
.sim-h{display:flex;justify-content:space-between;align-items:baseline;gap:10px;flex-wrap:wrap}
.sim-h b{font-size:.98rem}.sim-h span{color:var(--text2);font-size:.84rem}
.strip{display:grid;grid-template-columns:repeat(14,1fr);gap:2px;margin:10px 0 8px}
.strip div{height:18px;border-radius:3px;font-size:.6rem;display:flex;align-items:center;justify-content:center;font-weight:700}
.excerpt{font-size:.82rem;color:var(--text2);line-height:1.65}
.link{color:var(--blue);cursor:pointer;font-weight:600;font-size:.8rem;background:none;border:none;font-family:inherit}
.mech{display:flex;gap:10px;margin-bottom:10px;padding:12px 14px;background:var(--surface);border-radius:8px;border:1px solid var(--border);font-size:.84rem;line-height:1.6}
.path-box{background:var(--orange-l);border:1px solid var(--border);border-radius:var(--radius-sm);padding:18px 20px;margin-bottom:16px}
.path-box h4{font-size:.9rem;color:var(--orange);margin-bottom:6px}
.heat{width:100%;border-collapse:collapse;font-size:.78rem}
.heat th{padding:8px 10px;font-size:.72rem;color:var(--text2);border-bottom:1px solid var(--border);white-space:nowrap;cursor:pointer}
.heat th:first-child{text-align:left}
.heat td{padding:7px 10px;text-align:center;font-weight:600}
.heat td:first-child{text-align:left;font-weight:500}
.panel{background:var(--surface);border-radius:var(--radius);box-shadow:var(--shadow);padding:26px 28px;margin-bottom:22px}
@media(max-width:760px){.panel{padding:18px 16px}}
textarea,.inp{width:100%;font:inherit;font-size:.86rem;border:1.5px solid var(--border);border-radius:10px;padding:10px 12px;background:var(--surface);color:var(--text)}
textarea{min-height:130px;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.8rem}
.row{display:flex;gap:10px;flex-wrap:wrap;align-items:center;margin:10px 0}
.empty{color:var(--text2);text-align:center;padding:48px 16px}
.overflow{overflow-x:auto}
code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.82em;background:var(--surface2);padding:1px 5px;border-radius:5px}
</style>
</head>
<body>
<header class="header">
  <div class="logo">NeuroLoc-Δ · __NPROT__ proteine del cervello umano</div>
  <h1>Localizzazione subcellulare<br>nel cervello umano</h1>
  <p class="subtitle">Probabilità tarate per 9 compartimenti e 5 compartimenti del neurone, ripartizione delle copie, meccanismi di smistamento,
  proteine simili e varianti. Modello: __BEST__.</p>
  <div class="search">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.4"><circle cx="11" cy="11" r="7"/><path d="m20 20-3.5-3.5"/></svg>
    <input id="q" type="search" autocomplete="off" placeholder="Cerca per gene, nome o accession UniProt (es. MAPT, synuclein, P10636)">
    <div class="results" id="res"></div>
  </div>
  <div class="class-legend">
    <div class="legend-item"><div class="legend-dot" style="background:#4c9dff"></div>9 compartimenti generali</div>
    <div class="legend-item"><div class="legend-dot" style="background:#c58af2"></div>5 compartimenti del neurone (solo cellule giuste)</div>
    <div class="legend-item"><div class="legend-dot" style="background:#3fd6cc"></div>Multi-etichetta: una proteina può stare in più posti</div>
  </div>
</header>
<main>
  <nav class="modes">
    <button class="mode active" data-v="cerca">Scheda proteina</button>
    <button class="mode" data-v="nuova">Aggiungi una proteina</button>
    <button class="mode" data-v="confronto">Confronto</button>
    <button class="mode" data-v="guida">Come leggere</button>
  </nav>
  <section class="view active" id="v-cerca">
    <div class="chips" id="recent"></div>
    <div id="card"><div class="panel empty">Cerca una proteina nella barra in alto.</div></div>
  </section>
  <section class="view" id="v-nuova">
    <div class="panel">
      <div class="section-title">Aggiungi una proteina che non è nei dati</div>
      <div class="section-sub">Incolla la sequenza (anche in FASTA). L'analisi gira nel browser: segnali di smistamento nella sequenza, omologhi tra le
      __NPROT__ proteine del progetto, stima delle % per omologia e card_protein delle più simili.</div>
      <div class="row"><input class="inp" id="nn" placeholder="Nome (facoltativo)" style="max-width:280px"></div>
      <textarea id="ns" placeholder="MSEQUENZA..."></textarea>
      <div class="row"><button class="btn primary" id="go">Analizza</button><span class="note" id="nstat"></span></div>
      <div class="note">Predizione completa con la rete, la calibrazione e le percentuali: salva la sequenza in <code>data/esterni/nuove/nome.fasta</code>
      (intestazione <code>&gt;NOME ACCESSION</code>, accession facoltativa) e rilancia il notebook 03. La proteina comparirà qui come le altre, con l'etichetta "aggiunta da te".</div>
    </div>
    <div id="nout"></div>
  </section>
  <section class="view" id="v-confronto">
    <div class="panel">
      <div class="section-title">Confronto tra proteine</div>
      <div class="section-sub">Aggiungi proteine dalla loro scheda con "Aggiungi al confronto". Probabilità di presenza (%) per le 14 classi.</div>
      <div class="chips" id="pins"></div>
      <div class="overflow" id="heat"></div>
    </div>
    <div class="panel"><div class="section-title">Profilo dei meccanismi di smistamento</div>
      <div class="section-sub">Probabilità predette dalle teste ausiliarie della rete (0–100%).</div><div id="radar"></div></div>
  </section>
  <section class="view" id="v-guida">
    <div class="panel cardtxt">
      <div class="section-title">Come leggere la scheda</div>
      <p><strong>% probabilità.</strong> Quanto è probabile che la proteina stia in quel compartimento. È tarata: su 100 proteine al 70%, circa 70 ci stanno davvero.
      Non somma a 100, perché una proteina può stare in più posti (tau nell'assone e nel citoplasma). Unisce la sequenza, i partner molecolari e le misure già note.</p><br>
      <p><strong>Compartimenti del neurone.</strong> Assone, dendrite, presinapsi e postsinapsi hanno senso solo per le proteine espresse nei neuroni, la mielina solo
      per quelle degli oligodendrociti: altrimenti compare "—". Per una proteina aggiunta da te si mostrano sempre, da leggere come "se è espressa nei neuroni".</p><br>
      <p><strong>Ripartizione %.</strong> Quanta parte delle copie sta dove (somma 100). Citosol solubile e nucleo sono misurati in cellule umane quando esiste la misura
      (HEK293T, HeLa), altrimenti predetti; il resto si divide tra gli organelli secondo la presenza.</p><br>
      <p><strong>Dalla sola sequenza.</strong> Le localizzazioni che la rete prevede senza misure né partner: è la predizione confrontabile con DeepLoc 2.0.</p><br>
      <p><strong>Insieme ad alta sensibilità.</strong> Le classi sopra la soglia che tiene il 90% delle localizzazioni curate note. Sul test indipendente ne tiene circa l'81–85%: non è una garanzia.</p><br>
      <p><strong>Proteine simili.</strong> Le più vicine per embedding di sequenza (ESM-2 e ProtT5): omologhi, paraloghi e proteine della stessa famiglia.
      Le loro card_protein e localizzazioni sono il ragionamento per analogia: proteine simili tendono a stare negli stessi posti.</p><br>
      <p><strong>Varianti.</strong> Le varianti ClinVar più sospette secondo il modello delle varianti (notebook 03, sezione 7) e il meccanismo di smistamento che spostano di più.</p>
      <br><p class="note">AUC macro del modello sul test HPA indipendente (__NTEST__ proteine): __AUC__.</p>
    </div>
  </section>
</main>
<script>
const D = __DATI__;
const M = __META__;
const K = {}; D.cols.forEach((c, i) => K[c] = i);
const R = D.rows, NC = M.nC;
const COL = ['#0071e3','#5ac8fa','#5e5ce6','#64d2ff','#ff9500','#30d158','#ff6961','#bf5af2','#d4a106','#8944ab','#a55bd6','#9c27b0','#7b1fa2','#b39ddb'];
const MEC_TXT = {
  aux_nls: 'Il segnale di localizzazione nucleare è un punto fragile: sostituzioni nelle lisine e arginine dell\'NLS lasciano la proteina nel citoplasma (FUS nella SLA, mutazioni del PY-NLS).',
  aux_peptide_segnale: 'La proteina entra nella via secretoria con un peptide segnale: varianti che destabilizzano il ripiegamento la fanno trattenere nel reticolo endoplasmatico dal controllo di qualità.',
  aux_tm_1piu: 'Ha eliche transmembrana: una carica introdotta in un\'elica ne impedisce l\'inserzione in membrana (ΔΔG di Hessa) e la proteina resta nel RE o viene degradata (canalopatie, PLP1).',
  aux_tm_2piu: 'Proteina politopica di membrana: il ripiegamento corretto di più eliche è il passaggio critico; varianti destabilizzanti portano a ritenzione nel RE.',
  aux_transit_peptide: 'L\'import nel mitocondrio dipende dalla presequenza N-terminale anfipatica: varianti che ne tolgono le cariche positive bloccano l\'import (es. PINK1).',
  aux_ancora_lipidica: 'L\'ancoraggio alla membrana dipende da un lipide: perdere il residuo modificato (Gly N-terminale, Cys del CAAX) stacca la proteina dalla membrana.',
  aux_palmitoilazione: 'La palmitoilazione (come in PSD-95) porta la proteina alle membrane sinaptiche: perdere le cisteine N-terminali ne riduce il targeting sinaptico.',
  aux_glicoproteina: 'È glicosilata nella via secretoria: varianti che creano o tolgono sequon N-x-S/T cambiano ripiegamento e uscita dal RE.',
  aux_via_secretoria: 'Segue la via secretoria (RE → Golgi → membrana o lisosoma): ogni tappa ha un controllo di qualità che può trattenere le varianti mal ripiegate.',
  aux_nucleo_citoplasma: 'Fa la spola tra nucleo e citoplasma: l\'equilibrio tra import ed export è il bersaglio dei difetti di trasporto nucleo-citoplasmatico (TDP-43, FUS).',
  aux_import_organelli: 'Viene importata in un organello (mitocondrio o perossisoma): segnali di import alterati la lasciano nel citosol.'};
const $ = s => document.querySelector(s);
const esc = s => String(s == null ? '' : s).replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const f = (r, k) => r[K[k]];
function heat(v){ if (v === null || v === undefined) return 'var(--surface2)'; const t = Math.min(v/100, 1);
  const a = [[232,240,254],[76,157,255],[0,71,190]]; const u = t < .5 ? t/.5 : (t-.5)/.5; const p = t < .5 ? a[0] : a[1], q = t < .5 ? a[1] : a[2];
  return `rgb(${Math.round(p[0]+(q[0]-p[0])*u)},${Math.round(p[1]+(q[1]-p[1])*u)},${Math.round(p[2]+(q[2]-p[2])*u)})`; }
const heatTxt = v => (v !== null && v > 52) ? '#fff' : '#1d1d1f';
function bars(vals, from, to, colors){ let h = '';
  for (let c = from; c < to; c++){ const v = vals[c];
    if (v === null || v === undefined){ h += `<div class="bar na"><div class="nm">${M.cls[c]}</div><div class="track"></div><div class="pc">—</div></div>`; continue; }
    h += `<div class="bar${v >= 50 ? ' dom' : ''}"><div class="nm">${M.cls[c]}</div><div class="track"><div class="fill" style="width:${v}%;background:${colors[c]}"></div></div><div class="pc" style="color:${colors[c]}">${Math.round(v)}%</div></div>`; }
  return h; }
function tags(str, cls){ return (str || '').split(/;\s*/).filter(Boolean).map(t => `<span class="badge ${cls}">${esc(M.cls[M.cls_en.indexOf(t)] || t)}</span>`).join('') || '<span class="note">nessuna</span>'; }
function strip(p){ return '<div class="strip">' + p.map((v, c) => `<div title="${M.cls[c]}: ${v === null ? 'non applicabile' : v + '%'}" style="background:${heat(v)};color:${heatTxt(v)}">${v === null ? '' : v}</div>`).join('') + '</div>'; }
function cardHTML(t){ return esc(t).replace(/(^|\. )([A-Z][A-Za-z ]{2,40}(?: \([A-Za-z ]+\))?): /g, '$1<br><strong>$2:</strong> ').replace(/^<br>/, ''); }
function radar(series){ const n = M.mec.length, W = 460, cx = 230, cy = 205, rad = 150; let g = '';
  for (let k = 1; k <= 4; k++){ g += `<polygon points="${M.mec.map((_, i) => { const a = -Math.PI/2 + 2*Math.PI*i/n; return (cx + rad*k/4*Math.cos(a)) + ',' + (cy + rad*k/4*Math.sin(a)); }).join(' ')}" fill="none" stroke="var(--border)"/>`; }
  M.mec.forEach((m, i) => { const a = -Math.PI/2 + 2*Math.PI*i/n, x = cx + (rad + 14)*Math.cos(a), y = cy + (rad + 14)*Math.sin(a);
    g += `<line x1="${cx}" y1="${cy}" x2="${cx + rad*Math.cos(a)}" y2="${cy + rad*Math.sin(a)}" stroke="var(--border)"/><text x="${x}" y="${y}" font-size="10.5" fill="var(--text2)" text-anchor="${Math.cos(a) > .2 ? 'start' : Math.cos(a) < -.2 ? 'end' : 'middle'}" dominant-baseline="middle">${esc(m)}</text>`; });
  series.forEach((s, j) => { const col = COL[j % COL.length];
    g += `<polygon points="${s.v.map((v, i) => { const a = -Math.PI/2 + 2*Math.PI*i/n; return (cx + rad*v/100*Math.cos(a)) + ',' + (cy + rad*v/100*Math.sin(a)); }).join(' ')}" fill="${col}22" stroke="${col}" stroke-width="2.2"/>`; });
  const leg = series.map((s, j) => `<span class="badge" style="background:${COL[j % COL.length]}22;color:${COL[j % COL.length]}">${esc(s.n)}</span>`).join(' ');
  return `<svg viewBox="-40 0 ${W + 80} 420" style="width:100%;max-width:560px;display:block;margin:0 auto">${g}</svg><div class="badges" style="justify-content:center">${leg}</div>`; }
function simItem(j, score, lab){ const r = R[j]; const c = f(r, 'c') || '';
  return `<div class="sim"><div class="sim-h"><div><b>${esc(f(r, 'g'))}</b> <span>${esc(f(r, 'n'))}</span></div><span class="badge b-teal">${lab} ${score}%</span></div>
  ${strip(f(r, 'p'))}<div class="note" style="margin:0 0 6px"><strong>Localizzazioni:</strong> ${esc(f(r, 'l')) || '—'}</div>
  <div class="excerpt" id="ex${j}">${cardHTML(c.slice(0, 420))}${c.length > 420 ? '…' : ''}</div>
  <div class="row" style="margin:6px 0 0">${c.length > 420 ? `<button class="link" onclick="document.getElementById('ex${j}').innerHTML=cardHTML(R[${j}][K.c])">leggi tutta la card</button>` : ''}
  <button class="link" onclick="show(${j})">apri la scheda →</button></div></div>`; }
let recent = [], pins = [];
function show(i){ const r = R[i]; location.hash = encodeURIComponent(f(r, 'g'));
  recent = [i, ...recent.filter(x => x !== i)].slice(0, 10);
  $('#recent').innerHTML = recent.map(j => `<button class="chip" onclick="show(${j})">${esc(f(R[j], 'g'))}</button>`).join('');
  const p = f(r, 'p'), ex = f(r, 'ex') || [], neu = f(r, 'ne'), oli = f(r, 'ol'), src = f(r, 's');
  const ripCol = ['#0071e3','#5ac8fa','#5e5ce6','#64d2ff','#ff9500','#30d158','#ff6961','#bf5af2','#d4a106'];
  const rip = f(r, 'r').map((v, c) => v === null ? '' : `<div class="bar"><div class="nm">${M.rip[c]}${c < 2 ? ` <span class="note" style="color:${f(r, 'of').startsWith('misurata') ? 'var(--teal)' : 'var(--text3)'}">${f(r, 'of').startsWith('misurata') ? '✓ misurata' : 'predetta'}</span>` : ''}</div><div class="track" style="height:14px;border-radius:7px"><div class="fill" style="width:${v}%;background:${ripCol[c]};border-radius:7px"></div></div><div class="pc" style="color:${ripCol[c]}">${Math.round(v)}%</div></div>`).join('');
  const ms = (f(r, 'ms') || '').split(/;\s*/).filter(Boolean).map(t => `<div class="note">• ${esc(t)}</div>`).join('') || '<div class="note">Nessuna misura sperimentale tra le fonti del progetto.</div>';
  const mech = M.mec.map((m, k) => ({m, k: M.mec_key[k], v: f(r, 'm')[k]})).sort((a, b) => b.v - a.v);
  const strong = mech.filter(x => x.v >= 50);
  const pa = (f(r, 'pa') || []).map(g => `<button class="chip" onclick="findShow('${esc(g)}')">${esc(g)}</button>`).join('') || '<span class="note">Nessun partner noto in CORUM o OpenCell.</span>';
  const sims = f(r, 'si').map(([j, s]) => simItem(j, s, 'somiglianza')).join('');
  const dgr = (f(r, 'dg') || '').split('|').filter(Boolean).map(g => `<span class="badge b-red">${esc(g)}</span>`).join('');
  const vars = (f(r, 'va') || []).map(v => `<div class="mech"><div>🧬</div><div>${esc(v)}</div></div>`).join('') || '<div class="note">Nessuna variante ClinVar analizzata per questa proteina.</div>';
  const why = strong.length ? strong.map(x => `<div class="mech"><div>${x.v}%</div><div><strong>${esc(x.m)}.</strong> ${esc(MEC_TXT[x.k] || '')}</div></div>`).join('') : '<div class="note">Nessun meccanismo di smistamento predetto sopra il 50%: la localizzazione dipende soprattutto da partner e domini (proteina citosolica o periferica).</div>';
  const top = p.map((v, c) => ({v, c})).filter(x => x.v !== null).sort((a, b) => b.v - a.v).slice(0, 3).map(x => `${M.cls[x.c]} ${x.v}%`).join(' · ');
  $('#card').innerHTML = `<div class="protein-card">
  <div class="card-header"><div style="min-width:0;flex:1">
    <div class="card-gene">${esc(f(r, 'g'))}${f(r, 'u') ? ' · ' + esc(f(r, 'u')) : ''}</div>
    <div class="card-name">${esc(f(r, 'n')) || esc(f(r, 'g'))}</div>
    <div class="note" style="margin:0">${top}</div>
    <div class="badges"><span class="badge b-blue">${esc(src)}</span>
      ${neu === 1 ? '<span class="badge b-purple">espressa nei neuroni</span>' : ''}${oli === 1 ? '<span class="badge b-teal">espressa negli oligodendrociti</span>' : ''}
      ${f(r, 'bs') ? `<span class="badge b-grey">${esc(f(r, 'bs'))}</span>` : ''}<span class="badge b-grey">${f(r, 'nf')} fonti sperimentali</span>
      ${f(r, 'np') ? `<span class="badge b-red">${f(r, 'np')} varianti P/LP</span>` : ''}</div></div>
    <div class="side"><div class="info-label">Lunghezza</div><div class="info-value">${f(r, 'q').length} aa</div>
      <div class="info-label" style="margin-top:8px">Cervello (HPA)</div><div class="info-value">${ex[0] == null ? '—' : ex[0] + ' nTPM'}</div>
      <div class="row" style="justify-content:flex-end"><button class="btn" onclick="pin(${i})">Aggiungi al confronto</button>
      ${f(r, 'u') ? `<a class="btn" style="text-decoration:none" target="_blank" href="https://www.uniprot.org/uniprotkb/${esc(f(r, 'u'))}">UniProt ↗</a>` : ''}</div></div></div>
  <div class="tabs"><button class="tab active" data-p="pred">Predizione</button><button class="tab" data-p="bio">Scheda biologica</button>
    <button class="tab" data-p="sim">Proteine simili</button><button class="tab" data-p="pat">Patologia e varianti</button></div>
  <div class="pane active" id="p-pred"><div class="grid2">
    <div><div class="stitle">Compartimenti generali (9)</div>${bars(p, 0, NC, COL)}</div>
    <div><div class="stitle">Compartimenti del neurone (5)</div>${bars(p, NC, p.length, COL)}
      <div class="note">— = non applicabile: proteina non espressa in neuroni (assone, dendrite, sinapsi) o oligodendrociti (mielina).</div></div></div>
    <div class="box"><div class="stitle">Ripartizione delle copie (somma 100%)</div>${rip}</div>
    <div class="tags"><div class="stitle">Localizzazioni (P ≥ 50%)</div>${tags(f(r, 'l'), 'b-blue')}</div>
    <div class="tags"><div class="stitle">Dalla sola sequenza (soglie Jaccard)</div>${tags(f(r, 'sp'), 'b-grey')}</div>
    <div class="tags"><div class="stitle">Insieme ad alta sensibilità</div>${tags(f(r, 'cf'), 'b-grey')}</div>
    <div class="tags"><div class="stitle">Misure già note (fonte: classi)</div><div style="width:100%">${ms}</div></div></div>
  <div class="pane" id="p-bio">
    <div class="grid2"><div><div class="stitle">Espressione</div>
      <div class="note">Regioni cerebrali: <strong>${ex[0] == null ? '—' : ex[0] + ' nTPM'}</strong> · neuroni: <strong>${ex[1] == null ? '—' : ex[1] + ' nCPM'}</strong> ·
      glia: <strong>${ex[2] == null ? '—' : ex[2] + ' nCPM'}</strong> · oligodendrociti: <strong>${ex[3] == null ? '—' : ex[3] + ' nCPM'}</strong></div>
      <div class="stitle" style="margin-top:18px">Partner molecolari (CORUM, OpenCell)</div><div class="chips">${pa}</div></div>
      <div><div class="stitle">Meccanismi di smistamento predetti</div>${radar([{n: f(r, 'g'), v: f(r, 'm')}])}</div></div>
    <div class="box"><div class="stitle">card_protein</div><div class="cardtxt">${cardHTML(f(r, 'c'))}</div>
    <div class="note">Scheda senza parole di localizzazione: è un ingresso del modello, non la risposta.</div></div></div>
  <div class="pane" id="p-sim"><div class="section-sub">Le 8 proteine più vicine per embedding di sequenza (ESM-2 + ProtT5), con le loro card_protein e localizzazioni.
    La striscia colorata riporta le 14 % di probabilità (prima i 9 organelli, poi i 5 compartimenti del neurone).</div>${sims}</div>
  <div class="pane" id="p-pat">
    <div class="path-box"><h4>Malattie associate (UniProt)</h4><div class="cardtxt">${esc(f(r, 'dz')) || 'Nessuna malattia annotata.'}</div><div class="badges">${dgr}</div></div>
    <div class="stitle">Cosa succederebbe se fosse mislocalizzata: i meccanismi più forti di questa proteina</div>${why}
    <div class="stitle" style="margin-top:18px">Varianti ClinVar più sospette (modello delle varianti)</div>${vars}</div></div>`;
  document.querySelectorAll('.tab').forEach(b => b.onclick = () => { document.querySelectorAll('.tab').forEach(x => x.classList.toggle('active', x === b));
    document.querySelectorAll('.pane').forEach(x => x.classList.toggle('active', x.id === 'p-' + b.dataset.p)); });
  mode('cerca'); window.scrollTo({top: $('main').offsetTop - 10, behavior: 'smooth'}); }
function findShow(g){ const i = R.findIndex(r => f(r, 'g') === g); if (i >= 0) show(i); }
function search(q){ q = q.trim().toLowerCase(); if (!q) return [];
  const out = [];
  for (let i = 0; i < R.length; i++){ const g = String(f(R[i], 'g')).toLowerCase(), u = String(f(R[i], 'u')).toLowerCase(), n = String(f(R[i], 'n')).toLowerCase();
    const s = g === q || u === q ? 0 : g.startsWith(q) ? 1 : g.includes(q) ? 2 : n.includes(q) ? 3 : 9; if (s < 9) out.push([s, g.length, i]); }
  return out.sort((a, b) => a[0] - b[0] || a[1] - b[1]).slice(0, 30).map(x => x[2]); }
let hits = [], sel = 0;
$('#q').addEventListener('input', e => { hits = search(e.target.value); sel = 0; const box = $('#res');
  box.innerHTML = hits.map((i, k) => `<div class="res${k === 0 ? ' sel' : ''}" data-i="${i}"><b>${esc(f(R[i], 'g'))}</b><span>${esc(f(R[i], 'n'))}</span><em>${esc(f(R[i], 's'))}</em></div>`).join('') || '<div class="res"><span>Nessuna proteina trovata. Per una proteina nuova usa "Aggiungi una proteina".</span></div>';
  box.classList.toggle('open', e.target.value.trim().length > 0);
  box.querySelectorAll('[data-i]').forEach(d => d.onclick = () => { show(+d.dataset.i); box.classList.remove('open'); }); });
$('#q').addEventListener('keydown', e => { const items = [...document.querySelectorAll('#res [data-i]')]; if (!items.length) return;
  if (e.key === 'ArrowDown' || e.key === 'ArrowUp'){ e.preventDefault(); sel = (sel + (e.key === 'ArrowDown' ? 1 : -1) + items.length) % items.length;
    items.forEach((x, k) => x.classList.toggle('sel', k === sel)); items[sel].scrollIntoView({block: 'nearest'}); }
  if (e.key === 'Enter'){ show(+items[sel].dataset.i); $('#res').classList.remove('open'); } });
document.addEventListener('click', e => { if (!e.target.closest('.search')) $('#res').classList.remove('open'); });
function mode(v){ document.querySelectorAll('.mode').forEach(b => b.classList.toggle('active', b.dataset.v === v));
  document.querySelectorAll('.view').forEach(x => x.classList.toggle('active', x.id === 'v-' + v)); }
document.querySelectorAll('.mode').forEach(b => b.onclick = () => { mode(b.dataset.v); if (b.dataset.v === 'confronto') drawCompare(); });
function pin(i){ if (!pins.includes(i)) pins.push(i); pins = pins.slice(-10); mode('confronto'); drawCompare(); }
function drawCompare(){ $('#pins').innerHTML = pins.map(i => `<button class="chip" onclick="pins=pins.filter(x=>x!==${i});drawCompare()">${esc(f(R[i], 'g'))} ✕</button>`).join('') || '<span class="note">Nessuna proteina: apri una scheda e premi "Aggiungi al confronto".</span>';
  if (!pins.length){ $('#heat').innerHTML = ''; $('#radar').innerHTML = ''; return; }
  let h = `<table class="heat"><thead><tr><th>Compartimento</th>${pins.map(i => `<th onclick="show(${i})">${esc(f(R[i], 'g'))}</th>`).join('')}</tr></thead><tbody>`;
  M.cls.forEach((c, k) => { h += `<tr><td style="${k >= NC ? 'color:var(--purple)' : ''}">${k >= NC ? '🟣 ' : '🔵 '}${c}</td>` + pins.map(i => { const v = f(R[i], 'p')[k];
    return `<td style="background:${heat(v)};color:${heatTxt(v)}">${v === null ? '—' : v + '%'}</td>`; }).join('') + '</tr>'; });
  $('#heat').innerHTML = h + '</tbody></table>';
  $('#radar').innerHTML = radar(pins.map(i => ({n: f(R[i], 'g'), v: f(R[i], 'm')}))); }
// --- proteina nuova: analisi nel browser
const KD = {A:1.8,R:-4.5,N:-3.5,D:-3.5,C:2.5,Q:-3.5,E:-3.5,G:-0.4,H:-3.2,I:4.5,L:3.8,K:-3.9,M:1.9,F:2.8,P:-1.6,S:-0.8,T:-0.7,W:-0.9,Y:-1.3,V:4.2};
function signals(s){ const out = [], nt = s.slice(0, 30), ct40 = s.slice(-40), cnt = (t, re) => (t.match(re) || []).length;
  const kd = [...s].map(a => KD[a] || 0); let tm = 0, run = false;
  for (let i = 0; i + 19 <= s.length; i++){ let m = 0; for (let j = i; j < i + 19; j++) m += kd[j]; const hi = m / 19 >= 1.6; if (hi && !run) tm++; run = hi; }
  const ch = cnt(nt, /[KR]/g) - cnt(nt, /[DE]/g);
  if (/[AILMFVWC]{7,}/.test(s.slice(0, 40)) && ch <= 3) out.push(['Peptide segnale o ancora N-terminale', 'Nucleo idrofobico nei primi 40 residui: ingresso nel reticolo endoplasmatico e nella via secretoria.', 'b-green']);
  if (ch >= 3 && cnt(nt, /[DE]/g) <= 2) out.push(['Presequenza mitocondriale probabile', `N-terminale basico (carica netta +${ch} nei primi 30 residui) e povero di acidi: tipico dell'import nel mitocondrio.`, 'b-orange']);
  if (tm) out.push([`${tm} segmento/i idrofobico/i tipo elica TM`, 'Finestre di 19 residui con idrofobicità media ≥ 1,6 (Kyte-Doolittle): proteina di membrana.', 'b-blue']);
  if (/[KRHQSA][DENQ]EL$/.test(s)) out.push(['Segnale di ritenzione nel RE (KDEL)', 'C-terminale KDEL/HDEL: recuperata dal Golgi verso il reticolo endoplasmatico.', 'b-green']);
  if (/K.K..$|KK..$/.test(s)) out.push(['Motivo KKXX C-terminale', 'Ritenzione nel RE delle proteine di membrana.', 'b-green']);
  if (/[SACKNP][KRH][LMI]$/.test(s)) out.push(['PTS1 (perossisoma)', 'Tripeptide C-terminale riconosciuto da PEX5.', 'b-orange']);
  if (/K[KR].[KR]/.test(s) || /[KR]{2}.{10,12}[KR]{3}/.test(s)) out.push(['Possibile NLS', 'Gruppo di lisine/arginine monopartito o bipartito: import nel nucleo.', 'b-blue']);
  if (/[ST].[VIL]$/.test(s)) out.push(['Ligando PDZ di classe I', 'C-terminale S/T-x-V/I/L: ancoraggio a PSD-95 e alla densità postsinaptica.', 'b-purple']);
  if (/V[PA][IL]A..E[SD]D/.test(s)) out.push(['Motivo di legame all\'ankirina-G', 'Segmento iniziale dell\'assone (canali Nav e KCNQ).', 'b-purple']);
  if (/C[AILMV][AILMV].$/.test(s)) out.push(['Box CAAX', 'Prenilazione C-terminale: ancoraggio alla membrana.', 'b-teal']);
  if (/^MG[^EDRKHPFYW]..[STAGCN][^P]/.test(s)) out.push(['Miristoilazione N-terminale', 'Glicina N-terminale miristoilabile: associazione alle membrane.', 'b-teal']);
  if (cnt(s.slice(0, 20), /C/g) >= 2) out.push(['Cisteine N-terminali', 'Possibile palmitoilazione (come PSD-95, GAP-43): targeting alle membrane sinaptiche.', 'b-purple']);
  if (cnt(ct40, /Y..[LIMFV]/g) || cnt(ct40, /[DE]...L[LI]/g)) out.push(['Motivi YxxΦ / dileucina in coda', 'Smistamento endosoma-lisosoma; nei neuroni targeting somatodendritico.', 'b-grey']);
  if (cnt(s, /[PESQK]/g) / s.length > 0.35) out.push(['Ricca di regioni disordinate', 'Composizione P/E/S/Q/K elevata: domini disordinati, spesso nucleari o in condensati.', 'b-grey']);
  return out; }
function analyze(){ let s = $('#ns').value.replace(/^>.*$/gm, '').toUpperCase().replace(/[^A-Z]/g, ''), name = $('#nn').value.trim() || 'proteina nuova';
  if (s.length < 20){ $('#nstat').textContent = 'Servono almeno 20 aminoacidi.'; return; }
  $('#nstat').textContent = 'Analisi in corso…';
  setTimeout(() => { const qk = new Set(); for (let i = 0; i + 5 <= s.length; i++) qk.add(s.substr(i, 5));
    const sc = [];
    for (let j = 0; j < R.length; j++){ const t = f(R[j], 'q'); let h = 0; for (let i = 0; i + 5 <= t.length; i++) if (qk.has(t.substr(i, 5))) h++;
      const fr = Math.min(1, h / Math.max(1, Math.min(qk.size, t.length - 4))); if (h >= 8 && fr >= 0.02) sc.push([fr, j]); }
    sc.sort((a, b) => b[0] - a[0]);
    let neigh = sc.slice(0, 8), lab = '5-meri condivisi', weak = false;
    if (!neigh.length){ weak = true; lab = 'composizione';
      const AA = 'ACDEFGHIKLMNPQRSTVWY', vec = t => { const v = new Float32Array(400); for (let i = 0; i + 1 < t.length; i++){ const a = AA.indexOf(t[i]), b = AA.indexOf(t[i + 1]); if (a >= 0 && b >= 0) v[a*20 + b]++; }
        let n = 0; for (const x of v) n += x*x; n = Math.sqrt(n) || 1; return v.map(x => x/n); };
      const qv = vec(s); const cs = [];
      for (let j = 0; j < R.length; j++){ const v = vec(f(R[j], 'q')); let d = 0; for (let k = 0; k < 400; k++) d += qv[k]*v[k]; cs.push([d, j]); }
      neigh = cs.sort((a, b) => b[0] - a[0]).slice(0, 8); }
    const est = new Array(M.cls.length).fill(0), wt = new Array(M.cls.length).fill(0);
    neigh.forEach(([fr, j]) => { const w = weak ? Math.pow(fr, 8) : fr*fr; f(R[j], 'p').forEach((v, c) => { if (v !== null){ est[c] += w*v; wt[c] += w; } }); });
    const pe = est.map((e, c) => wt[c] ? Math.round(e/wt[c]) : null);
    const sig = signals(s).map(x => `<div class="mech"><div><span class="badge ${x[2]}">${esc(x[0])}</span></div><div>${esc(x[1])}</div></div>`).join('') || '<div class="note">Nessun segnale di smistamento evidente: probabile proteina citosolica o localizzata tramite partner.</div>';
    $('#nout').innerHTML = `<div class="protein-card"><div class="card-header"><div><div class="card-gene">analisi rapida nel browser</div><div class="card-name">${esc(name)}</div>
      <div class="badges"><span class="badge b-grey">${s.length} aa</span><span class="badge ${weak ? 'b-orange' : 'b-teal'}">${weak ? 'nessun omologo: stima debole per composizione' : sc.length + ' omologhi nel progetto'}</span></div></div></div>
      <div class="pane active"><div class="stitle">Segnali di smistamento nella sequenza</div>${sig}
      <div class="grid2" style="margin-top:18px"><div><div class="stitle">Stima per omologia: compartimenti generali</div>${bars(pe, 0, NC, COL)}</div>
      <div><div class="stitle">Stima per omologia: compartimenti del neurone</div>${bars(pe, NC, pe.length, COL)}
      <div class="note">Media pesata delle % delle proteine più simili${weak ? ' (solo composizione aminoacidica: molto incerta)' : ''}. Per la rete completa usa il notebook 03.</div></div></div>
      <div class="stitle" style="margin-top:22px">Proteine più simili del progetto e loro card_protein</div>
      ${neigh.map(([fr, j]) => simItem(j, Math.round(100*fr), weak ? 'composizione' : '5-meri')).join('')}</div></div>`;
    $('#nstat').textContent = 'Fatto.'; }, 30); }
$('#go').onclick = analyze;
const h0 = decodeURIComponent(location.hash.slice(1)); if (h0) findShow(h0);
</script>
</body>
</html>'''
pagina = (PAGINA.replace('__DATI__', DATI.replace('</', '<\\/')).replace('__META__', json.dumps(META, ensure_ascii=False))
          .replace('__NPROT__', f'{len(R):,}'.replace(',', '.')).replace('__BEST__', cfg['BEST']).replace('__NTEST__', str(cfg['n_test']))
          .replace('__AUC__', f"{META['auc_best']:.3f}"))
(DATA / 'neuroloc_delta.html').write_text(pagina, encoding='utf-8')
print(f"Pagina scritta: {DATA / 'neuroloc_delta.html'} ({len(pagina) / 1e6:.1f} MB). Si apre con doppio clic, senza server né internet.")

## 12 · Contesto: il test HOU di Wefers et al. 2026 (confronto non appaiato)
Wefers et al. (Nature Methods, 2026) hanno costruito un benchmark rigoroso per la localizzazione umana dalla sequenza:
3.814 proteine umane (test **HOU**) con etichette confermate da almeno due banche dati tra HPA, OpenCell e UniProt e
identità inferiore al 40% con il loro training. Le stesse proteine però sono quasi tutte nei **nostri** dati: per noi
sono proteine di training, e le possiamo valutare solo con le predizioni fuori fold. Il confronto quindi **non è
appaiato** e va letto come contesto, non come gara:
- **modelli diversi su dati diversi**: le loro AUC vengono dai loro fold di training, le nostre dai nostri. Gli zip di
  Zenodo contengono i modelli, non le predizioni per proteina sul test, quindi un confronto appaiato non è possibile;
- **solo sequenza**: si usa la rete senza scheda testuale (`P_oof_seq`, ablazione `abl_senza_card` dello 02). La
  scheda porta conoscenza dei curatori (sezione 4b dello 02) e le fusioni con le misure userebbero le banche dati da
  cui vengono le etichette HOU;
- **4 classi su 7 del livello 3**: nucleo, citosol, membrana plasmatica, mitocondrio. Il loro "sistema di endomembrane"
  contiene anche vescicole ed endosomi, che il nostro schema non ha: non è confrontabile e resta fuori, come nucleoli e
  citoscheletro;
- **AUC, non AP**: l'AP dipende dalla prevalenza, diversa tra l'intero test e i nostri sottoinsiemi. Si riporta la
  nostra AUC con l'intervallo di confidenza bootstrap e si dice se il valore pubblicato cade sotto, dentro o sopra.
  "Dentro" vuol dire **parità**, non superiorità: così va scritto.

Un solo insieme: le proteine HOU che abbiamo. Sono già tutte nel perimetro cerebrale senza sangue (lo 01 tiene solo
quelle), quindi il vecchio secondo sottoinsieme era identico al primo. Uscite:
`data/confronto_wefers.xlsx`, `data/roc_wefers.png`.


In [ ]:
# 12 · contesto: Wefers et al. 2026, test HOU (confronto non appaiato, solo sequenza, 4 classi)
import requests
import yaml

WEF = DATA / 'esterni' / 'wefers'
WEF.mkdir(parents=True, exist_ok=True)
GH = 'https://raw.githubusercontent.com/CellProfiling/seq2loc_benchmark/main/'
ZEN = 'https://zenodo.org/records/'
# file di testo dal repository del paper (contenuto fisso), Zenodo come riserva. Gli zip dei modelli non servono: non
# contengono predizioni per proteina sul test
SCARICA = [('hou_testset.csv', [GH + 'datasets/final/hou_testset.csv', ZEN + '18435290/files/hou_testset.csv?download=1']),
           ('hierarchical_label_set.yaml', [GH + 'datasets/final/hierarchical_label_set.yaml',
                                            ZEN + '18435290/files/hierarchical_label_set.yaml?download=1']),
           ('test_avgs.csv', [GH + 'Benchmark-Models/results/test_avgs.csv']),
           ('test_perclass_level3.csv', [GH + 'Benchmark-Models/results/test_perclass_level3.csv'])]
# ogni file si scarica una volta sola (prima in .part, poi rinominato): un download interrotto non lascia file a metà
for nome, urls in SCARICA:
    for _ in range(int(not (WEF / nome).exists())):
        for u in urls:
            try:
                with requests.get(u, stream=True, timeout=120) as r_:
                    r_.raise_for_status()
                    with open(WEF / f'{nome}.part', 'wb') as fh:
                        for blocco in r_.iter_content(1 << 20):
                            fh.write(blocco)
                (WEF / f'{nome}.part').rename(WEF / nome)
                print(f'scaricato {nome} da {u.split("/")[2]}')
                break
            except Exception as err:
                print(f'{nome}: {u.split("/")[2]} non raggiungibile ({type(err).__name__})')

# etichette HOU al livello 3: una colonna 0/1 per classe, nell'ordine del loro file yaml
# lettura sempre in UTF-8 (sostituendo i byte non validi): la codifica predefinita del sistema operativo può non esserlo
TXT = dict(encoding='utf-8', encoding_errors='replace')
hou = pd.read_csv(WEF / 'hou_testset.csv', **TXT)
hou['acc'] = hou['uniprot_id'].astype(str).str.split('-').str[0]
L3 = yaml.safe_load((WEF / 'hierarchical_label_set.yaml').read_bytes().decode('utf-8', errors='replace'))['level3']
Y_hou = hou['level3'].str.get_dummies(sep=';').reindex(columns=L3, fill_value=0)
MAPPA = {'nucleus': 'Nucleus', 'cytosol': 'Cytoplasm', 'plasma-membrane': 'Cell membrane', 'mitochondria': 'Mitochondrion'}
CMP = list(MAPPA)

# predizioni di sola sequenza: fuori fold per il training, media dei fold per il test HPA
P_seq = z['P_oof_seq'].astype(np.float64).copy()
P_seq[test_rows] = z['P_hpa_seq']
# le proteine HOU nei nostri dati, per accession canonica e, in mancanza, per sequenza identica
pid_acc = pd.Series(np.arange(N_PROT), index=proteins['uniprot_id'].astype(str).str.split('-').str[0])
pid_seq = pd.Series(np.arange(N_PROT), index=proteins['sequence'].astype(str))
pid_h = hou['acc'].map(pid_acc.groupby(level=0).first()).fillna(hou['sequence'].map(pid_seq.groupby(level=0).first()))
pid_i = pid_h.fillna(0).astype(int).to_numpy()
ce_l = pid_h.notna().to_numpy() & np.isfinite(P_seq[pid_i, 0])
print(f"Test HOU: {len(hou):,} proteine | nei nostri dati: {pid_h.notna().sum():,}, di cui di training (deeploc o "
      f"umane aggiunte): {(pid_h.notna().to_numpy() & np.isin(src[pid_i], ['deeploc', 'human_extra'])).sum():,} | "
      f"con predizione di sola sequenza fuori fold o di test: {ce_l.sum():,}")

# AUC per classe con IC bootstrap: ogni ricampionamento è un vettore di conteggi (multinomiale) sulle proteine. L'AUC
# pesata si calcola in forma chiusa sulle proteine ordinate per punteggio: per ogni positivo il peso dei negativi con
# punteggio più basso, più metà di quelli a pari punteggio
B_H = 1000
ce_l = ce_l & z['valutabile'][pid_i]                # fuori le proteine con un'omologa curata in un altro fold
SOTTO = {'tutte le HOU che abbiamo': ce_l}
righe_h, AUC_B = [], {}
for nome_s, m_s in SOTTO.items():
    P_s = P_seq[np.ix_(pid_i[m_s], [CLASSES.index(MAPPA[c]) for c in CMP])]
    Y_s = Y_hou[CMP].to_numpy()[m_s]
    W_b = rng.multinomial(len(Y_s), np.full(len(Y_s), 1 / len(Y_s)), size=B_H).astype(np.float32)
    col_b = []
    for j in range(len(CMP)):
        o_ = np.argsort(P_s[:, j])
        s_, y_ = P_s[o_, j], Y_s[o_, j]
        wp, wn = W_b[:, o_] * y_, W_b[:, o_] * (1 - y_)
        cn = np.c_[np.zeros(B_H), np.cumsum(wn, 1)]
        lo, hi = np.searchsorted(s_, s_, 'left'), np.searchsorted(s_, s_, 'right')
        col_b.append((wp * (cn[:, lo] + 0.5 * (cn[:, hi] - cn[:, lo]))).sum(1) / (wp.sum(1) * wn.sum(1)))
    AUC_B[nome_s] = np.c_[np.stack(col_b, 1), np.stack(col_b, 1).mean(1)]                 # [ricampionamento, classe + macro]
    auc_s = [roc_auc_score(Y_s[:, j], P_s[:, j]) for j in range(len(CMP))]
    righe_h.append(pd.DataFrame({'sottoinsieme': nome_s, 'classe': CMP + ['macro'], 'proteine': len(Y_s),
                                 'positivi': list(Y_s.sum(0)) + [Y_s.sum()], 'AUC NeuroLoc-Δ (sola sequenza)': auc_s + [np.mean(auc_s)],
                                 'IC 2,5%': np.nanquantile(AUC_B[nome_s], 0.025, 0), 'IC 97,5%': np.nanquantile(AUC_B[nome_s], 0.975, 0)}))
nostro = pd.concat(righe_h, ignore_index=True)
print('\nNeuroLoc-Δ, sola sequenza, sulle proteine HOU (fuori fold):')
print(nostro.round(3).to_string(index=False))

# valori pubblicati sull'intero test HOU (3.814 proteine), stesse 4 classi: non appaiati. Si dice solo se cadono sotto,
# dentro o sopra il nostro IC 95% sul primo sottoinsieme (il più grande)
pcl = pd.read_csv(WEF / 'test_perclass_level3.csv', **TXT).pivot_table(index='model', columns='label', values='rocauc_perclass')
pub_cmp = pcl.reindex(columns=CMP).dropna()
pub_cmp['macro'] = pub_cmp[CMP].mean(axis=1)
ic_t = nostro[nostro['sottoinsieme'] == 'tutte le HOU che abbiamo'].set_index('classe')[['IC 2,5%', 'IC 97,5%']]
pos_ic = pd.DataFrame({c: np.array(['sotto', 'dentro', 'sopra'])[np.searchsorted(ic_t.loc[c].to_numpy(), pub_cmp[c].to_numpy())]
                       for c in CMP + ['macro']}, index=pub_cmp.index)
print("\nAUC pubblicate da Wefers et al. sull'intero test HOU, stesse 4 classi:")
print(pub_cmp.round(3).to_string())
print('\nValore pubblicato rispetto al nostro IC 95% (tutte le HOU che abbiamo; non appaiato, solo indicativo):')
print(pos_ic.to_string())
pub = pd.read_csv(WEF / 'test_avgs.csv', **TXT)
pub = pub[pub['level'] == 3].set_index('model')[['macro_ap', 'rocauc_macro', 'jaccard_macro', 'f1_macro']]
pub.loc['ProtT5-MHA (Tab. 1 del paper)', 'macro_ap'] = 0.709                  # il loro modello migliore non è nel file
print("\nMetriche pubblicate sull'intero test, tutte e 7 le classi del livello 3 (non confrontabili con le nostre 4):")
print(pub.round(3).to_string())
with pd.ExcelWriter(DATA / 'confronto_wefers.xlsx') as xw:
    nostro.round(4).to_excel(xw, sheet_name='NeuroLoc-Δ sola sequenza', index=False)
    pub_cmp.round(4).to_excel(xw, sheet_name='pubblicato, 4 classi')
    pos_ic.to_excel(xw, sheet_name='pubblicato vs nostro IC')
    pub.round(4).to_excel(xw, sheet_name='pubblicato, 7 classi')

# curve ROC: una per classe e la macro (media delle TPR su una griglia comune di FPR). I valori
# pubblicati, sull'intero test, sono solo in legenda
m_c = SOTTO['tutte le HOU che abbiamo']
P_c = P_seq[np.ix_(pid_i[m_c], [CLASSES.index(MAPPA[c]) for c in CMP])]
Y_c = Y_hou[CMP].to_numpy()[m_c]
auc_c = nostro[nostro['sottoinsieme'] == 'tutte le HOU che abbiamo'].set_index('classe')['AUC NeuroLoc-Δ (sola sequenza)']
grid_f = np.linspace(0, 1, 201)
fig, axs = plt.subplots(2, 3, figsize=(15, 9.5), facecolor='white')
for j, (c, ax) in enumerate(zip(CMP + ['macro'], axs.ravel())):
    cur = [roc_curve(Y_c[:, k], P_c[:, k]) for k in [[j], range(len(CMP))][int(c == 'macro')]]
    ax.plot(grid_f, np.mean([np.interp(grid_f, f_, t_) for f_, t_, _ in cur], 0), color='#c0392b', lw=2.2,
            label=f'NeuroLoc-Δ, sola sequenza: AUC {auc_c[c]:.3f}')
    for mp in pub_cmp.index:
        ax.plot([], [], ' ', label=f'{mp}, pubblicato sull\'intero test: {pub_cmp.loc[mp, c]:.3f}')
    ax.plot([0, 1], [0, 1], ':', color='#999999')
    ax.set(title=c, xlabel='Falsi positivi (FPR)', ylabel='Veri positivi (TPR)')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)
axs.ravel()[-1].axis('off')
fig.suptitle(f'Test HOU di Wefers et al. 2026: {m_c.sum():,} proteine, 4 classi del livello 3 (non appaiato)', fontsize=14)
fig.tight_layout()
fig.savefig(DATA / 'roc_wefers.png', dpi=150, facecolor='white')
plt.show()
